# Burgers 1D: audited U-Net, direct CNO, and F-FNO baselines

This notebook evaluates exactly three controls on the same Burgers trajectories,
spatial grid, saved time snapshots, nested sample sizes, validation set, test
set, and absolute \(L^2(\Omega\times T)\) implementation used by the AWFNO
benchmark.

| Label in results | Training map | Test-time protocol |
|---|---|---|
| **U-Net-AR** | Every adjacent pair \(u_n\mapsto u_{n+1}\) | Recursive rollout from the exact \(u_0\); no exact future state |
| **CNO-Direct** | \((u_0,x)\mapsto(u_1,\ldots,u_{N_t-1})\) | One non-autoregressive model call |
| **F-FNO** | Every adjacent pair \(u_n\mapsto u_{n+1}\) | Recursive rollout from the exact \(u_0\); no exact future state |

Important provenance:

- U-Net-AR uses the PDEBench `UNet1d` encoder/decoder topology. Its convolution
  padding is changed from zero to circular because this Burgers problem is
  periodic. Training is a one-step adjacent-pair protocol; this is not claimed
  to reproduce PDEBench's smoother \(\nu=1\), 10-frame-input benchmark.
- CNO-Direct uses the upstream `CNO1d_simplified/CNO1d.py` architecture and
  training defaults (mean L1, AdamW, weight decay \(10^{-8}\), StepLR
  \(15,0.5\)). Only `out_dim` is set to \(N_t-1\) so one call represents the
  complete future trajectory.
- FourierFlow publishes 2D/3D F-FNO but no separate 1D implementation. F-FNO
  here is the competitive one-spatial-axis adaptation already used in the
  earlier Burgers notebook:
  shared Fourier weights, width 64, 24 layers, 16 modes, factor 4, weight
  normalization, state noise, MSE training, AdamW, gradient clipping, and
  cosine decay over epochs. It receives the same periodic coordinate features
  \(x,\sin(2\pi x),\cos(2\pi x)\) as AWFNO. This faster variant uses batch
  size 64. It is a compute/quality compromise and is not numerically identical
  to the earlier competitive batch-16 run.
- FourierFlow's published turbulent-flow configuration validates only a
  10-step recursive horizon from a later exact state. This notebook instead
  selects checkpoints using the complete 200-step rollout from \(u_0\), which
  is the common and substantially stricter AWFNO evaluation protocol.

Pinned source commits are written into every result manifest:

- PDEBench: `4ff3e3a4aa1561721b5571fa3a048a0a463e0568`
- ConvolutionalNeuralOperator: `6e765198aa02b56352e0a3437104b9d9e337176e`
- FourierFlow: `82d1689e02c95f7656a7eaa1d88d0ab407b2483d`

For paper use, retain the complete JSON manifest, `.npz` rollouts, checkpoints,
training curves, per-time error plot, parameter counts, and timing report saved
to Google Drive. Do not select epochs or hyperparameters using test error.

## 1. Install the small runtime dependencies

PyTorch is already provided by the Colab GPU runtime.

In [ ]:
%pip -q install h5py matplotlib


## 2. Mount Drive and configure the experiment

Change only `DATA_PATH` and, if desired, `OUTPUT_ROOT`. The default configuration first runs only the N=25 pilot. After checking that the selected F-FNO validation rollout is finite, set `PILOT_ONLY=False` to run every nested size. Cached checkpoints make the notebook resumable.

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

import os

DATA_PATH = "/content/gdrive/MyDrive/anus/1D_Burgers_Sols_Nu0.001.hdf5"
OUTPUT_ROOT = "/content/gdrive/MyDrive/anus/burgers_paper_baselines_v1"
PILOT_ONLY = False

CONFIG = {
    # Data and disjoint nested split.
    "BENCHMARK_SUITE": "paper_baselines",
    "DATA_PATH": DATA_PATH,
    "OUTPUT_ROOT": OUTPUT_ROOT,
    "ALL_SAMPLE_SIZES": "25,50,100,200,400,800",
    "RUN_SAMPLE_SIZES": "25" if PILOT_ONLY else "25,50,100,200,400,800",
    "N_VAL": "50",
    "N_TEST": "100",
    "SPLIT_MODE": "contiguous",
    "SPLIT_SEED": "2026",
    "SEED": "1234",
    "TARGET_NX": "512",
    "TIME_SUBSAMPLE": "1",
    "LOW_RAM_HDF5": "1",

    # Common training and audit controls.
    "EPOCHS": "200",
    "FORCE_RETRAIN": "0",
    "RUN_EXACT_HASH_OVERLAP_CHECK": "1",
    "RUN_DISTANCE_OVERLAP_CHECK": "0",
    "USE_TRAPEZOIDAL_TIME_L2": "1",
    "INCLUDE_T0_IN_FULL_L2": "1",

    # Correct one-step autoregressive U-Net.
    "UNET_AR_FEATURES": "32",
    "UNET_AR_BATCH_SIZE": "64",
    "UNET_AR_LR": "0.001",
    "UNET_AR_WEIGHT_DECAY": "0.0001",
    "UNET_AR_VAL_PAIR_INTERVAL": "10",
    "UNET_AR_VAL_ROLLOUT_INTERVAL": "10",
    "UNET_AR_GRAD_CLIP": "1.0",

    # Original upstream CNO1d, used non-autoregressively.
    "CNO_LAYERS": "4",
    "CNO_RES_BLOCKS": "4",
    "CNO_NECK_BLOCKS": "4",
    "CNO_CHANNEL_MULTIPLIER": "16",
    "CNO_USE_BATCH_NORM": "0",
    "CNO_DIRECT_BATCH_SIZE": "10",
    "CNO_DIRECT_LR": "0.001",
    "CNO_DIRECT_WEIGHT_DECAY": "1e-8",
    "CNO_DIRECT_VAL_INTERVAL": "1",

    # FourierFlow-style one-step F-FNO.
    "FFNO_WIDTH": "64",
    "FFNO_LAYERS": "24",
    "FFNO_MODES": "16",
    "FFNO_FACTOR": "4",
    "FFNO_FF_LAYERS": "2",
    "FFNO_SHARE_FOURIER_WEIGHT": "1",
    "FFNO_WEIGHT_NORM": "1",
    "FFNO_NOISE_STD": "0.01",
    "FFNO_LR": "0.0025",
    "FFNO_WEIGHT_DECAY": "0.0001",
    "FFNO_BATCH_SIZE": "64",
    "FFNO_GRAD_CLIP": "1.0",
    "FFNO_PAPER_VAL_PAIR_INTERVAL": "10",
    "FFNO_PAPER_VAL_ROLLOUT_INTERVAL": "10",

    # Reuse compatible completed checkpoints. The incompatible batch-512
    # paper-v1 F-FNO checkpoints are never reused.
    "REUSE_V1_NON_FFNO": "1",
    "REUSE_COMPETITIVE_FFNO": "0",

    # Saved evidence.
    "SAVE_PLOTS": "1",
    "SHOW_PLOTS": "0",
    "MAKE_TRAINING_CURVE_PLOTS": "1",
    "MAKE_L2_PER_TIME_PLOT": "1",
    "MAKE_PER_TEST_L2_PLOTS": "0",
    "MAKE_SOLUTION_PLOTS": "1",
    "MAX_TEST_CASES_TO_PLOT": "10",
    "PLOT_TIME_INDICES": "25,50,final",
    "SMOKE_TEST": "0",
}

os.environ.update(CONFIG)
print("Configured output:", OUTPUT_ROOT)
print("Configured training sizes:", CONFIG["RUN_SAMPLE_SIZES"])


## 3. Audited self-contained implementation

Run this cell once to define the data loader, metrics, all three architectures, training protocols, validation selection, evaluation, plots, checkpoints, and manifests. The paper suite is independent of the legacy experiment suites retained in the source.

In [ ]:
"""Self-contained paper-baseline implementation: U-Net-AR, CNO-Direct, and F-FNO."""
import os

import gc

import re

import json

import copy

import time

import math

import random

import hashlib

from pathlib import Path

from collections import OrderedDict

import numpy as np

import matplotlib.pyplot as plt

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

def env_bool(name, default=False):
    value = os.environ.get(name, None)
    if value is None:
        return bool(default)
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}

def env_int(name, default):
    value = os.environ.get(name, None)
    return int(value) if value not in (None, "") else int(default)

def env_float(name, default):
    value = os.environ.get(name, None)
    return float(value) if value not in (None, "") else float(default)

def env_list_int(name, default_list):
    value = os.environ.get(name, None)
    if value is None or str(value).strip() == "":
        return list(default_list)
    return [int(x.strip()) for x in str(value).split(",") if x.strip()]

SEED = env_int("SEED", 1234)

DATA_PATH = Path(os.environ.get(
    "DATA_PATH",
    "/content/gdrive/MyDrive/anus/1D_Burgers_Sols_Nu0.001.hdf5",
))

OUTPUT_ROOT = Path(os.environ.get(
    "OUTPUT_ROOT",
    "/content/gdrive/MyDrive/anus/burgers_awfno_outputs",
))

TARGET_NX = env_int("TARGET_NX", 512)

TIME_SUBSAMPLE = env_int("TIME_SUBSAMPLE", 1)

HDF5_SOLUTION_KEY = os.environ.get("HDF5_SOLUTION_KEY", "") or None

HDF5_X_KEY = os.environ.get("HDF5_X_KEY", "") or None

HDF5_T_KEY = os.environ.get("HDF5_T_KEY", "") or None

SPLIT_MODE = os.environ.get("SPLIT_MODE", "contiguous")  # "contiguous" or "random"

SPLIT_SEED = env_int("SPLIT_SEED", 2026)

ALL_SAMPLE_SIZES = env_list_int("ALL_SAMPLE_SIZES", [25, 50, 100, 200, 400, 800])

SAMPLE_SIZES = env_list_int("RUN_SAMPLE_SIZES", ALL_SAMPLE_SIZES)

N_TRAIN_POOL = max(ALL_SAMPLE_SIZES)

N_VAL = env_int("N_VAL", 50)

N_TEST = env_int("N_TEST", 100)

LOW_RAM_HDF5 = env_bool("LOW_RAM_HDF5", True)

WINDOW_K = env_int("WINDOW_K", 5)

WINDOW_STRIDE = env_int("WINDOW_STRIDE", 5)

N_TIME_MODES = env_int("N_TIME_MODES", 5)

BATCH_SIZE = env_int("BATCH_SIZE", 128)

EPOCHS = env_int("EPOCHS", 200)

LR = env_float("LR", 1.0e-3)

WEIGHT_DECAY = env_float("WEIGHT_DECAY", 1.0e-6)

WIDTH = env_int("WIDTH", 32)

N_LAYERS = env_int("N_LAYERS", 4)

MODES_X = env_int("MODES_X", 16)

LOCAL_RADIUS = env_int("LOCAL_RADIUS", 3)

FINAL_LOSS_WEIGHT = env_float("FINAL_LOSS_WEIGHT", 1.0)

ENFORCE_MEAN = env_bool("ENFORCE_MEAN", False)

USE_LIBRARY_FNO = env_bool("USE_LIBRARY_FNO", True)

FORCE_RETRAIN = env_bool("FORCE_RETRAIN", False)

RUN_EXACT_HASH_OVERLAP_CHECK = env_bool("RUN_EXACT_HASH_OVERLAP_CHECK", True)

RUN_DISTANCE_OVERLAP_CHECK = env_bool("RUN_DISTANCE_OVERLAP_CHECK", False)

OVERLAP_TOL = env_float("OVERLAP_TOL", 1.0e-10)

SAVE_PLOTS = env_bool("SAVE_PLOTS", True)

SHOW_PLOTS = env_bool("SHOW_PLOTS", False)

MAKE_TRAINING_CURVE_PLOTS = env_bool("MAKE_TRAINING_CURVE_PLOTS", True)

MAKE_L2_PER_TIME_PLOT = env_bool("MAKE_L2_PER_TIME_PLOT", True)

MAKE_PER_TEST_L2_PLOTS = env_bool("MAKE_PER_TEST_L2_PLOTS", False)

MAKE_SOLUTION_PLOTS = env_bool("MAKE_SOLUTION_PLOTS", True)

MAKE_BEST_WORST_PLOTS = env_bool("MAKE_BEST_WORST_PLOTS", True)

N_BEST_WORST = env_int("N_BEST_WORST", 2)

MAX_TEST_CASES_TO_PLOT = os.environ.get("MAX_TEST_CASES_TO_PLOT", "10")

MAX_TEST_CASES_TO_PLOT = None if str(MAX_TEST_CASES_TO_PLOT).lower() == "none" else int(MAX_TEST_CASES_TO_PLOT)

PLOT_DPI = env_int("PLOT_DPI", 140)

L2_CHUNK_SIZE = env_int("L2_CHUNK_SIZE", 50)

USE_TRAPEZOIDAL_TIME_L2 = env_bool("USE_TRAPEZOIDAL_TIME_L2", True)

INCLUDE_T0_IN_FULL_L2 = env_bool("INCLUDE_T0_IN_FULL_L2", True)

_plot_times_raw = os.environ.get("PLOT_TIME_INDICES", "25,50,final")

PLOT_TIME_INDICES_MANUAL = []

for item in _plot_times_raw.split(","):
    item = item.strip()
    if not item:
        continue
    PLOT_TIME_INDICES_MANUAL.append("final" if item.lower() == "final" else int(item))

EXPERIMENT_NAME = os.environ.get("EXPERIMENT_NAME", "burgers_awfno_hdf5_sample_sweep")

BENCHMARK_SUITE = os.environ.get("BENCHMARK_SUITE", "paper_baselines").strip().lower()

FFNO_WIDTH = env_int("FFNO_WIDTH", 64)

FFNO_LAYERS = env_int("FFNO_LAYERS", 24)

FFNO_MODES = env_int("FFNO_MODES", 16)

FFNO_FACTOR = env_int("FFNO_FACTOR", 4)

FFNO_FF_LAYERS = env_int("FFNO_FF_LAYERS", 2)

FFNO_SHARE_FOURIER_WEIGHT = env_bool("FFNO_SHARE_FOURIER_WEIGHT", True)

FFNO_WEIGHT_NORM = env_bool("FFNO_WEIGHT_NORM", True)

FFNO_NOISE_STD = env_float("FFNO_NOISE_STD", 0.01)

FFNO_LR = env_float("FFNO_LR", 2.5e-3)

FFNO_WEIGHT_DECAY = env_float("FFNO_WEIGHT_DECAY", 1.0e-4)

FFNO_BATCH_SIZE = env_int("FFNO_BATCH_SIZE", 64)

FFNO_VAL_PAIR_INTERVAL = env_int("FFNO_VAL_PAIR_INTERVAL", 10)

FFNO_VAL_ROLLOUT_INTERVAL = env_int("FFNO_VAL_ROLLOUT_INTERVAL", 50)

FFNO_GRAD_CLIP = env_float("FFNO_GRAD_CLIP", 1.0)

REUSE_V1_NON_FFNO = env_bool("REUSE_V1_NON_FFNO", True)

REUSE_COMPETITIVE_FFNO = env_bool("REUSE_COMPETITIVE_FFNO", True)

CNO_LAYERS = env_int("CNO_LAYERS", 4)

CNO_RES_BLOCKS = env_int("CNO_RES_BLOCKS", 4)

CNO_NECK_BLOCKS = env_int("CNO_NECK_BLOCKS", 4)

CNO_CHANNEL_MULTIPLIER = env_int("CNO_CHANNEL_MULTIPLIER", 16)

CNO_USE_BATCH_NORM = env_bool("CNO_USE_BATCH_NORM", False)

PAPER_BASELINE_VERSION = "paper-three-baselines-v4-batch64"

PAPER_BASELINE_NAMES = ["U-Net-AR", "CNO-Direct", "F-FNO"]

UNET_AR_FEATURES = env_int("UNET_AR_FEATURES", 32)

UNET_AR_BATCH_SIZE = env_int("UNET_AR_BATCH_SIZE", 64)

UNET_AR_LR = env_float("UNET_AR_LR", 1.0e-3)

UNET_AR_WEIGHT_DECAY = env_float("UNET_AR_WEIGHT_DECAY", 1.0e-4)

UNET_AR_VAL_PAIR_INTERVAL = env_int("UNET_AR_VAL_PAIR_INTERVAL", 10)

UNET_AR_VAL_ROLLOUT_INTERVAL = env_int("UNET_AR_VAL_ROLLOUT_INTERVAL", 10)

UNET_AR_GRAD_CLIP = env_float("UNET_AR_GRAD_CLIP", 1.0)

CNO_DIRECT_BATCH_SIZE = env_int("CNO_DIRECT_BATCH_SIZE", 10)

CNO_DIRECT_LR = env_float("CNO_DIRECT_LR", 1.0e-3)

CNO_DIRECT_WEIGHT_DECAY = env_float("CNO_DIRECT_WEIGHT_DECAY", 1.0e-8)

CNO_DIRECT_VAL_INTERVAL = env_int("CNO_DIRECT_VAL_INTERVAL", 1)

CNO_DIRECT_GRAD_CLIP = env_float("CNO_DIRECT_GRAD_CLIP", 1.0)

FFNO_PAPER_VAL_PAIR_INTERVAL = env_int(
    "FFNO_PAPER_VAL_PAIR_INTERVAL", FFNO_VAL_PAIR_INTERVAL
)

FFNO_PAPER_VAL_ROLLOUT_INTERVAL = env_int(
    "FFNO_PAPER_VAL_ROLLOUT_INTERVAL", 1
)

UNET_START_CHANNELS = env_int("UNET_START_CHANNELS", 8)

TEMPORAL_CNO_LR = env_float("TEMPORAL_CNO_LR", 5.0e-4)

TEMPORAL_CNO_WEIGHT_DECAY = env_float("TEMPORAL_CNO_WEIGHT_DECAY", 1.0e-6)

TEMPORAL_CNO_BATCH_SIZE = env_int("TEMPORAL_CNO_BATCH_SIZE", 32)

TEMPORAL_CNO_PAIRS_PER_TRAJECTORY = env_int(
    "TEMPORAL_CNO_PAIRS_PER_TRAJECTORY", 32
)

TEMPORAL_CNO_VAL_INTERVAL = env_int("TEMPORAL_CNO_VAL_INTERVAL", 10)

TEMPORAL_CNO_RELATIVE_L1 = env_bool("TEMPORAL_CNO_RELATIVE_L1", True)

TEMPORAL_CNO_FILM_HIDDEN = env_int("TEMPORAL_CNO_FILM_HIDDEN", 128)

PDEBENCH_UNET_FEATURES = env_int("PDEBENCH_UNET_FEATURES", 32)

PDEBENCH_UNET_PUSHFORWARD_STEPS = env_int(
    "PDEBENCH_UNET_PUSHFORWARD_STEPS", 20
)

PDEBENCH_UNET_LR = env_float("PDEBENCH_UNET_LR", 1.0e-3)

PDEBENCH_UNET_WEIGHT_DECAY = env_float(
    "PDEBENCH_UNET_WEIGHT_DECAY", 1.0e-4
)

PDEBENCH_UNET_BATCH_SIZE = env_int("PDEBENCH_UNET_BATCH_SIZE", 8)

PDEBENCH_UNET_VAL_INTERVAL = env_int("PDEBENCH_UNET_VAL_INTERVAL", 10)

PDEBENCH_UNET_USE_STATE_NORMALIZATION = env_bool(
    "PDEBENCH_UNET_USE_STATE_NORMALIZATION", True
)

PDEBENCH_UNET_AVERAGE_PUSHFORWARD_LOSS = env_bool(
    "PDEBENCH_UNET_AVERAGE_PUSHFORWARD_LOSS", True
)

PDEBENCH_UNET_GRAD_CLIP = env_bool("PDEBENCH_UNET_GRAD_CLIP", True)

PDEBENCH_DIRECT_BATCH_SIZE = env_int("PDEBENCH_DIRECT_BATCH_SIZE", 50)

PDEBENCH_DIRECT_VAL_INTERVAL = env_int("PDEBENCH_DIRECT_VAL_INTERVAL", 1)

PDEBENCH_DIRECT_FINAL_LOSS_WEIGHT = env_float(
    "PDEBENCH_DIRECT_FINAL_LOSS_WEIGHT", 1.0
)

BASELINE_INPUT_CHANNELS = 4

TEMPORAL_CNO_INPUT_CHANNELS = BASELINE_INPUT_CHANNELS + 1

COMPETITIVE_BASELINE_VERSION = "five-baselines-fast-v4"

OFFICIAL_BASELINE_NAMES = [
    "F-FNO",
    "CNO",
    "U-Net",
    "CNO-Time",
    "U-Net-PF",
]

SMOKE_TEST = env_bool("SMOKE_TEST", False)

CACHE_PROTOCOL_VERSION = (
    PAPER_BASELINE_VERSION
    if BENCHMARK_SUITE in {"paper_baselines", "paper"}
    else COMPETITIVE_BASELINE_VERSION
)

CACHE_ROOT = OUTPUT_ROOT / (
    f"cache_{EXPERIMENT_NAME}_"
    f"suite{BENCHMARK_SUITE}_"
    f"version{CACHE_PROTOCOL_VERSION}_"
    f"sizes{'-'.join(map(str, ALL_SAMPLE_SIZES))}_"
    f"K{WINDOW_K}_M{N_TIME_MODES}_W{WIDTH}_mx{MODES_X}_R{LOCAL_RADIUS}"
)

MODEL_DIR = CACHE_ROOT / "models"

PLOT_DIR = CACHE_ROOT / "plots"

DATA_DIR = CACHE_ROOT / "data"

RESULTS_DIR = CACHE_ROOT / "results"

for d in [MODEL_DIR, PLOT_DIR, DATA_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAPER_V1_CACHE_ROOT = OUTPUT_ROOT / (
    f"cache_{EXPERIMENT_NAME}_"
    "suitepaper_baselines_"
    "versionpaper-three-baselines-v1_"
    f"sizes{'-'.join(map(str, ALL_SAMPLE_SIZES))}_"
    f"K{WINDOW_K}_M{N_TIME_MODES}_W{WIDTH}_mx{MODES_X}_R{LOCAL_RADIUS}"
)

PAPER_V1_MODEL_DIR = PAPER_V1_CACHE_ROOT / "models"

plt.rcdefaults()

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "figure.dpi": 120,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "-",
    "axes.spines.top": True,
    "axes.spines.right": True,
})

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

    # GPU speed settings. TF32 is useful on Ampere+ GPUs and harmless elsewhere.
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PIN_MEMORY = device.type == "cuda"

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

x_grid_np = None

t_grid = None

Nx = None

Nt = None

DX = None

DT_SNAPSHOT = 1.0

BASE_TAG = "uninitialized"

def set_geometry_from_arrays(U_shape, x_grid, t=None):
    global x_grid_np, t_grid, Nx, Nt, DX, DT_SNAPSHOT

    if len(U_shape) != 3:
        raise ValueError(f"Expected U shape (N,Nx,Nt). Got {U_shape}")

    _, Nx, Nt = U_shape
    x_grid_np = np.asarray(x_grid, dtype=np.float32)

    if len(x_grid_np) != Nx:
        raise ValueError(f"x_grid length {len(x_grid_np)} does not match Nx={Nx}")

    DX = float(np.mean(np.diff(x_grid_np.astype(np.float64)))) if Nx > 1 else 1.0

    if t is None:
        t_grid = np.arange(Nt, dtype=np.float32)
        DT_SNAPSHOT = 1.0
    else:
        t_grid = np.asarray(t, dtype=np.float32)
        if len(t_grid) != Nt:
            print("Warning: t_grid length mismatch. Using snapshot index as time grid.")
            t_grid = np.arange(Nt, dtype=np.float32)
            DT_SNAPSHOT = 1.0
        elif Nt > 1:
            DT_SNAPSHOT = float(np.mean(np.diff(t_grid.astype(np.float64))))
        else:
            DT_SNAPSHOT = 1.0

    print("\nGeometry")
    print("--------")
    print("N, Nx, Nt   :", U_shape[0], Nx, Nt)
    print("x range     :", float(x_grid_np.min()), float(x_grid_np.max()))
    print("DX          :", DX)
    print("DT_SNAPSHOT :", DT_SNAPSHOT)

def short_hash(obj):
    txt = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.md5(txt.encode("utf-8")).hexdigest()[:10]

def clean_state_dict(sd):
    sd = copy.deepcopy(sd)
    if isinstance(sd, dict) and "_metadata" in sd:
        del sd["_metadata"]
    if hasattr(sd, "_metadata"):
        try:
            del sd._metadata
        except Exception:
            pass
    return OrderedDict((k, v) for k, v in sd.items() if k != "_metadata")

def save_model_checkpoint(path, model, meta=None):
    torch.save({"state_dict": clean_state_dict(model.state_dict()), "meta": meta or {}}, path)

def load_model_checkpoint(path, model):
    payload = torch.load(path, map_location=device)
    model.load_state_dict(clean_state_dict(payload["state_dict"]), strict=True)
    return payload.get("meta", {})

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def reset_seed_for_sample_size(n_train):
    """Make each sample-size run reproducible and independent of run order."""
    run_seed = int(SEED) + 100_000 + int(n_train)
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(run_seed)
    return run_seed

def safe_name(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    return text.strip("_") or "item"

def to_device(x):
    return x.to(device, non_blocking=PIN_MEMORY)

def _require_h5py():
    try:
        import h5py  # noqa: F401
    except ImportError as exc:
        raise ImportError("Install h5py with: pip install h5py") from exc

def _list_hdf5_datasets(h5file):
    datasets = []
    def visitor(name, obj):
        if hasattr(obj, "shape") and obj.shape is not None:
            datasets.append((name, tuple(obj.shape), str(getattr(obj, "dtype", ""))))
    h5file.visititems(visitor)
    return datasets

def _find_first_matching_key(h5file, candidates):
    all_names = []
    def visitor(name, obj):
        if hasattr(obj, "shape") and obj.shape is not None:
            all_names.append(name)
    h5file.visititems(visitor)

    lower_to_name = {name.lower(): name for name in all_names}
    for cand in candidates:
        if cand.lower() in lower_to_name:
            return lower_to_name[cand.lower()]

    for cand in candidates:
        c = cand.lower()
        for name in all_names:
            n = name.lower()
            if n.endswith("/" + c) or n == c:
                return name
    return None

def _auto_find_solution_key(h5file):
    """Find the solution dataset without reading large arrays into RAM."""
    preferred = ["tensor", "U", "u", "solution", "solutions", "data", "sols", "fields", "trajectories"]
    key = _find_first_matching_key(h5file, preferred)
    if key is not None:
        dset = h5file[key]
        if len(dset.shape) in (3, 4):
            return key

    candidates = []
    for name, shape, dtype in _list_hdf5_datasets(h5file):
        if len(shape) in (3, 4) and np.prod(shape) > 0:
            candidates.append((name, shape, dtype, int(np.prod(shape))))
    if not candidates:
        raise KeyError("Could not automatically find a 3D/4D solution dataset in the HDF5 file.")
    candidates.sort(key=lambda item: item[-1], reverse=True)
    return candidates[0][0]

def _auto_find_grid_key(h5file, kind):
    if kind == "x":
        candidates = ["x", "x_grid", "x-coordinate", "x_coordinate", "space", "grid/x", "coordinates/x"]
    elif kind == "t":
        candidates = ["t", "time", "t_grid", "t-coordinate", "t_coordinate", "grid/t", "coordinates/t"]
    else:
        raise ValueError(kind)

    key = _find_first_matching_key(h5file, candidates)
    if key is None:
        return None
    arr = np.asarray(h5file[key])
    return key if arr.ndim == 1 else None

def _squeeze_solution_array(arr):
    arr = np.asarray(arr, dtype=np.float32)
    while arr.ndim > 3 and 1 in arr.shape:
        axis = arr.shape.index(1)
        arr = np.squeeze(arr, axis=axis)
    if arr.ndim != 3:
        raise ValueError(f"Expected a 3D array after squeezing singleton channels. Got {arr.shape}")
    return arr.astype(np.float32, copy=False)

def _convert_solution_to_nxt(arr, x=None, t=None):
    arr = _squeeze_solution_array(arr)
    shape = arr.shape
    axes = list(range(3))
    x_len = None if x is None else int(len(x))
    t_len = None if t is None else int(len(t))

    if x_len is not None and t_len is not None:
        x_axes = [a for a in axes if shape[a] == x_len]
        t_axes = [a for a in axes if shape[a] == t_len]
        for ax_x in x_axes:
            for ax_t in t_axes:
                if ax_x == ax_t:
                    continue
                ax_n = [a for a in axes if a not in (ax_x, ax_t)][0]
                return np.transpose(arr, (ax_n, ax_x, ax_t)).astype(np.float32)

    if x_len is not None:
        x_axes = [a for a in axes if shape[a] == x_len]
        if x_axes:
            ax_x = x_axes[0]
            remaining = [a for a in axes if a != ax_x]
            if ax_x == 1:
                ax_n, ax_t = 0, 2
            elif ax_x == 2:
                ax_n, ax_t = 0, 1
            else:
                ax_n, ax_t = remaining[0], remaining[1]
            return np.transpose(arr, (ax_n, ax_x, ax_t)).astype(np.float32)

    if t_len is not None:
        t_axes = [a for a in axes if shape[a] == t_len]
        if t_axes:
            ax_t = t_axes[0]
            remaining = [a for a in axes if a != ax_t]
            if ax_t == 2:
                ax_n, ax_x = 0, 1
            elif ax_t == 1:
                ax_n, ax_x = 0, 2
            else:
                ax_n, ax_x = remaining[0], remaining[1]
            return np.transpose(arr, (ax_n, ax_x, ax_t)).astype(np.float32)

    return arr.astype(np.float32)

def _read_solution_subset_low_ram(dset, n_rows_needed):
    """
    Read only the trajectories needed for the configured split.

    This assumes the HDF5 solution dataset uses the standard PDEBench convention
    where the first axis is the trajectory/sample axis. This is true for the
    usual Burgers files, e.g. tensor.shape = (N, T, Nx) or (N, Nx, T).
    """
    total_rows = int(dset.shape[0])
    n_read = min(int(n_rows_needed), total_rows)

    if SPLIT_MODE == "random":
        rng = np.random.default_rng(SPLIT_SEED)
        chosen = rng.permutation(total_rows)[:n_read]
        order = np.argsort(chosen)
        chosen_sorted = chosen[order]
        raw_sorted = np.asarray(dset[chosen_sorted, ...], dtype=np.float32)
        inv_order = np.argsort(order)
        raw = raw_sorted[inv_order]
        print(f"LOW_RAM_HDF5: read {n_read}/{total_rows} randomly selected trajectories from HDF5.")
    else:
        raw = np.asarray(dset[:n_read, ...], dtype=np.float32)
        print(f"LOW_RAM_HDF5: read first {n_read}/{total_rows} trajectories from HDF5.")

    return raw

def load_hdf5_1d_dataset(path):
    _require_h5py()
    import h5py

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"HDF5 data file not found: {path}\n"
            "Set DATA_PATH to the correct local path before running."
        )

    n_rows_needed = N_TRAIN_POOL + N_VAL + N_TEST

    with h5py.File(path, "r") as f:
        print("\nHDF5 datasets")
        print("-------------")
        for name, shape, dtype in _list_hdf5_datasets(f):
            print(f"  {name}: shape={shape}, dtype={dtype}")

        solution_key = HDF5_SOLUTION_KEY or _auto_find_solution_key(f)
        x_key = HDF5_X_KEY or _auto_find_grid_key(f, "x")
        t_key = HDF5_T_KEY or _auto_find_grid_key(f, "t")

        print("\nSelected HDF5 keys")
        print("------------------")
        print("solution:", solution_key)
        print("x       :", x_key)
        print("t       :", t_key)

        dset = f[solution_key]
        print("solution shape:", tuple(dset.shape), "dtype:", dset.dtype)
        print("LOW_RAM_HDF5:", LOW_RAM_HDF5)
        print("rows needed for split:", n_rows_needed)

        x = None if x_key is None else np.asarray(f[x_key], dtype=np.float32).reshape(-1)
        t = None if t_key is None else np.asarray(f[t_key], dtype=np.float32).reshape(-1)

        if LOW_RAM_HDF5:
            if len(dset.shape) not in (3, 4):
                raise ValueError(f"Expected 3D/4D solution dataset. Got shape {tuple(dset.shape)}")
            raw = _read_solution_subset_low_ram(dset, n_rows_needed)
        else:
            print("WARNING: LOW_RAM_HDF5=False; reading the entire solution dataset into RAM.")
            raw = np.asarray(dset, dtype=np.float32)

    U = _convert_solution_to_nxt(raw, x=x, t=t)
    del raw
    gc.collect()

    if x is None:
        x = np.arange(U.shape[1], dtype=np.float32)
    if t is None:
        t = np.arange(U.shape[2], dtype=np.float32)

    if TIME_SUBSAMPLE > 1:
        U = U[:, :, ::TIME_SUBSAMPLE].copy()
        t = t[::TIME_SUBSAMPLE].copy()
        gc.collect()

    if not np.all(np.isfinite(U)):
        raise ValueError("Loaded solution array contains NaN or Inf values.")

    U = U.astype(np.float32, copy=False)
    x = x.astype(np.float32, copy=False)
    t = t.astype(np.float32, copy=False)

    print("\nLoaded data")
    print("-----------")
    print("U shape:", U.shape, " internal layout = (N,Nx,Nt)")
    print("x shape:", x.shape)
    print("t shape:", t.shape)
    print("U memory GB:", U.nbytes / 1024**3)
    print("U min/max:", float(U.min()), float(U.max()))

    return U, x, t

def downsample_space_stride(U, x, target_nx):
    if target_nx is None or int(target_nx) <= 0:
        return U, x

    target_nx = int(target_nx)
    U = np.asarray(U, dtype=np.float32)
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    N, Nx0, Nt0 = U.shape

    if len(x) != Nx0:
        raise ValueError(f"len(x)={len(x)} does not match Nx={Nx0}")
    if target_nx > Nx0:
        raise ValueError(f"TARGET_NX={target_nx} is larger than current Nx={Nx0}")
    if Nx0 % target_nx != 0:
        raise ValueError(f"Stride downsampling requires Nx={Nx0} divisible by TARGET_NX={target_nx}")

    factor = Nx0 // target_nx
    U_down = U[:, ::factor, :]
    x_down = x[::factor]

    print("\nSpatial downsampling")
    print("--------------------")
    print("original U shape:", U.shape)
    print("target Nx       :", target_nx)
    print("stride factor   :", factor)
    print("new U shape     :", U_down.shape)

    return U_down.astype(np.float32), x_down.astype(np.float32)

def make_train_val_test_splits(U_all):
    needed = N_TRAIN_POOL + N_VAL + N_TEST
    if U_all.shape[0] < needed:
        raise ValueError(
            f"Need at least {needed} trajectories: train_pool={N_TRAIN_POOL}, "
            f"val={N_VAL}, test={N_TEST}, but data has {U_all.shape[0]}. "
            "Reduce ALL_SAMPLE_SIZES/RUN_SAMPLE_SIZES/N_VAL/N_TEST."
        )

    ids = np.arange(U_all.shape[0], dtype=np.int64)
    if SPLIT_MODE == "random":
        rng = np.random.default_rng(SPLIT_SEED)
        ids = rng.permutation(ids)
    elif SPLIT_MODE != "contiguous":
        raise ValueError(f"Unknown SPLIT_MODE={SPLIT_MODE!r}")

    selected = ids[:needed]
    train_ids = selected[:N_TRAIN_POOL]
    val_ids = selected[N_TRAIN_POOL:N_TRAIN_POOL + N_VAL]
    test_ids = selected[N_TRAIN_POOL + N_VAL:N_TRAIN_POOL + N_VAL + N_TEST]

    U_train_pool = U_all[train_ids]
    U_val = U_all[val_ids]
    U_test = U_all[test_ids]

    print("\nData split")
    print("----------")
    print("split mode       :", SPLIT_MODE)
    print("U_train_pool     :", U_train_pool.shape)
    print("U_val            :", U_val.shape)
    print("U_test           :", U_test.shape)
    print("train ids first  :", train_ids[:5])
    print("val ids first    :", val_ids[:5])
    print("test ids first   :", test_ids[:5])

    return U_train_pool, train_ids, U_val, val_ids, U_test, test_ids

def save_split_dataset_npz(path, U_train_pool, train_ids, U_val, val_ids, U_test, test_ids, x, t):
    np.savez_compressed(
        path,
        U_train_pool=U_train_pool,
        train_ids=train_ids,
        U_val=U_val,
        val_ids=val_ids,
        U_test=U_test,
        test_ids=test_ids,
        x_grid=x,
        t_grid=t,
        sample_sizes=np.array(ALL_SAMPLE_SIZES, dtype=np.int64),
        run_sample_sizes=np.array(SAMPLE_SIZES, dtype=np.int64),
        data_path=np.array(str(DATA_PATH)),
    )
    print("Saved split dataset:", path)

def trajectory_hashes(U):
    U32 = np.ascontiguousarray(U.astype(np.float32, copy=False))
    return [hashlib.sha1(U32[i].tobytes()).hexdigest() for i in range(U32.shape[0])]

def check_exact_hash_overlap(U_a, U_b, name_a, name_b):
    h_a = set(trajectory_hashes(U_a))
    h_b = set(trajectory_hashes(U_b))
    inter = h_a.intersection(h_b)
    print(f"\nExact hash overlap check {name_a} vs {name_b}")
    print("------------------------------------------")
    print("duplicates:", len(inter))
    if inter:
        raise RuntimeError(f"Exact duplicate trajectory found between {name_a} and {name_b}.")

def check_dataset_overlap(U_a, U_b, name_a="A", name_b="B", tol=1.0e-10):
    A = U_a.reshape(U_a.shape[0], -1)
    B = U_b.reshape(U_b.shape[0], -1)
    min_dist = float("inf")
    min_pair = None
    for i in range(A.shape[0]):
        d = np.linalg.norm(B - A[i][None, :], axis=1)
        j = int(np.argmin(d))
        if d[j] < min_dist:
            min_dist = float(d[j])
            min_pair = (i, j)
    print(f"\nDistance overlap check {name_a} vs {name_b}: min distance={min_dist:.6e}, pair={min_pair}")
    if min_dist < tol:
        raise RuntimeError(f"Possible duplicate trajectory between {name_a} and {name_b}.")
    return min_dist, min_pair

def _time_weights_np(nt):
    w = np.ones(nt, dtype=np.float64)
    if USE_TRAPEZOIDAL_TIME_L2 and nt > 1:
        w[0] = 0.5
        w[-1] = 0.5
    return w

def _time_weights_torch(nt, device_, dtype):
    w = torch.ones(nt, device=device_, dtype=dtype)
    if USE_TRAPEZOIDAL_TIME_L2 and nt > 1:
        w[0] = 0.5
        w[-1] = 0.5
    return w

def abs_l2_window_batch(pred, true):
    w = _time_weights_torch(pred.shape[-1], pred.device, pred.dtype).view(1, 1, -1)
    err2 = (pred - true) ** 2
    return torch.sqrt(torch.sum(err2 * w, dim=(1, 2)) * DX * DT_SNAPSHOT + 1.0e-30)

def abs_l2_field_batch(pred, true):
    return torch.sqrt(torch.sum((pred - true) ** 2, dim=1) * DX + 1.0e-30)

def l2_per_timestep_np(pred, true, chunk_size=L2_CHUNK_SIZE):
    if pred.shape != true.shape:
        raise ValueError(f"Shape mismatch: pred {pred.shape}, true {true.shape}")
    n, nx, nt = pred.shape
    if nx != Nx:
        raise ValueError(f"Geometry mismatch: array has Nx={nx}, global Nx={Nx}")

    out = np.zeros((n, nt), dtype=np.float64)
    for i0 in range(0, n, chunk_size):
        i1 = min(i0 + chunk_size, n)
        err = pred[i0:i1].astype(np.float32, copy=False) - true[i0:i1].astype(np.float32, copy=False)
        out[i0:i1] = np.sqrt(np.sum(err * err, axis=1, dtype=np.float64) * DX + 1.0e-30)
        del err
        gc.collect()
    return out

def abs_l2_full_traj_np(pred, true):
    if INCLUDE_T0_IN_FULL_L2:
        pred_use, true_use = pred, true
    else:
        pred_use, true_use = pred[:, :, 1:], true[:, :, 1:]
    l2_t = l2_per_timestep_np(pred_use, true_use)
    w = _time_weights_np(l2_t.shape[1]).reshape(1, -1)
    return np.sqrt(np.sum(w * l2_t ** 2, axis=1) * DT_SNAPSHOT + 1.0e-30)

def abs_l2_final_np(pred, true):
    return l2_per_timestep_np(pred, true)[:, -1]

def rollout_l2_metrics(pred, true):
    full = abs_l2_full_traj_np(pred, true)
    final = abs_l2_final_np(pred, true)
    l2_t = l2_per_timestep_np(pred, true)
    return {
        "full_l2_mean": float(np.mean(full)),
        "full_l2_std": float(np.std(full)),
        "full_l2_median": float(np.median(full)),
        "final_l2_mean": float(np.mean(final)),
        "final_l2_std": float(np.std(final)),
        "final_l2_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "all_full_l2": full,
        "all_final_l2": final,
        "all_l2_t": l2_t,
    }

class FullTrajectoryDataset1D(Dataset):
    def __init__(self, U):
        U = np.asarray(U, dtype=np.float32)
        if U.ndim != 3:
            raise ValueError(f"Expected U with shape (N,Nx,Nt). Got {U.shape}")
        if U.shape[-1] < 2:
            raise ValueError("Need at least two saved time snapshots for full-trajectory prediction.")
        self.u0 = torch.tensor(U[:, :, 0], dtype=torch.float32)
        self.U_future = torch.tensor(U[:, :, 1:], dtype=torch.float32)

    def __len__(self):
        return self.u0.shape[0]

    def __getitem__(self, i):
        return {"u0": self.u0[i], "U_future": self.U_future[i]}

class MarkovPairDataset1D(Dataset):
    """All adjacent pairs from a set of trajectories, without materializing copies."""

    def __init__(self, U):
        U = np.asarray(U, dtype=np.float32)
        if U.ndim != 3:
            raise ValueError(f"Expected U with shape (N,Nx,Nt). Got {U.shape}")
        if U.shape[-1] < 2:
            raise ValueError("Need at least two snapshots to construct Markov pairs.")
        self.U = torch.as_tensor(U, dtype=torch.float32)
        self.n_trajectories, self.nx, self.nt = self.U.shape
        self.steps_per_trajectory = self.nt - 1

    def __len__(self):
        return self.n_trajectories * self.steps_per_trajectory

    def __getitem__(self, i):
        trajectory = i // self.steps_per_trajectory
        step = i % self.steps_per_trajectory
        return {
            "u_start": self.U[trajectory, :, step],
            "u_next": self.U[trajectory, :, step + 1],
        }

class GaussianNormalizer:
    def __init__(self, x, dims=None, eps=1.0e-6):
        self.eps = eps
        if dims is None:
            self.mean = x.mean()
            self.std = x.std()
        else:
            self.mean = x.mean(dim=dims, keepdim=True)
            self.std = x.std(dim=dims, keepdim=True)

    def encode(self, x):
        return (x - self.mean) / (self.std + self.eps)

    def decode(self, x):
        return x * (self.std + self.eps) + self.mean

    def to(self, device_):
        self.mean = self.mean.to(device_)
        self.std = self.std.to(device_)
        return self

class FFNOWLinear(nn.Linear):
    """Linear layer used by FourierFlow's feed-forward blocks."""

    def __init__(self, in_features, out_features, bias=True, use_weight_norm=False):
        super().__init__(in_features, out_features, bias=bias)
        if use_weight_norm:
            nn.utils.weight_norm(self)

class FFNOFeedForward(nn.Module):
    """FourierFlow FeedForward, with the same expansion and residual output."""

    def __init__(self, dim, factor=4, n_layers=2, use_weight_norm=True,
                 layer_norm=False, dropout=0.0):
        super().__init__()
        layers = []
        for i in range(n_layers):
            in_dim = dim if i == 0 else dim * factor
            out_dim = dim if i == n_layers - 1 else dim * factor
            layers.append(nn.Sequential(
                FFNOWLinear(in_dim, out_dim, use_weight_norm=use_weight_norm),
                nn.Dropout(dropout),
                nn.ReLU(inplace=True) if i < n_layers - 1 else nn.Identity(),
                nn.LayerNorm(out_dim) if layer_norm and i == n_layers - 1 else nn.Identity(),
            ))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class FFNOSpectralLayer1d(nn.Module):
    """
    One-dimensional specialization of FourierFlow's separable spectral layer.

    FourierFlow applies one 1D transform per spatial axis and sums the axis
    outputs. Burgers has one spatial axis, so there is exactly one transform.
    """

    def __init__(self, width, n_modes, fourier_weight=None, factor=4,
                 n_ff_layers=2, use_weight_norm=True):
        super().__init__()
        self.width = int(width)
        self.n_modes = int(n_modes)
        if fourier_weight is None:
            weight = torch.empty(self.width, self.width, self.n_modes, 2)
            nn.init.xavier_normal_(weight, gain=0.1)
            self.fourier_weight = nn.Parameter(weight)
        else:
            self.fourier_weight = fourier_weight
        self.backcast_ff = FFNOFeedForward(
            self.width,
            factor=factor,
            n_layers=n_ff_layers,
            use_weight_norm=use_weight_norm,
            layer_norm=False,
            dropout=0.0,
        )

    def forward_fourier(self, x):
        # x: (B, Nx, width), matching FourierFlow's channels-last convention.
        x_cf = x.permute(0, 2, 1).contiguous()
        batch, _, nx = x_cf.shape
        x_ft = torch.fft.rfft(x_cf, dim=-1, norm="ortho")
        out_ft = x_ft.new_zeros(batch, self.width, nx // 2 + 1)
        modes = min(self.n_modes, nx // 2 + 1)
        weight = torch.view_as_complex(self.fourier_weight[:, :, :modes].contiguous())
        out_ft[:, :, :modes] = torch.einsum(
            "bim,iom->bom", x_ft[:, :, :modes], weight
        )
        out = torch.fft.irfft(out_ft, n=nx, dim=-1, norm="ortho")
        return out.permute(0, 2, 1).contiguous()

    def forward(self, x):
        return self.backcast_ff(self.forward_fourier(x))

class OfficialCNOLReLU1d(nn.Module):
    def __init__(self, in_size, out_size):
        super().__init__()
        self.in_size = int(in_size)
        self.out_size = int(out_size)
        self.act = nn.LeakyReLU()

    def forward(self, x):
        x = F.interpolate(
            x.unsqueeze(2),
            size=(1, 2 * self.in_size),
            mode="bicubic",
            antialias=True,
        )
        x = self.act(x)
        x = F.interpolate(
            x,
            size=(1, self.out_size),
            mode="bicubic",
            antialias=True,
        )
        return x[:, :, 0]

class OfficialCNOBlock1d(nn.Module):
    def __init__(self, in_channels, out_channels, in_size, out_size, use_bn=True):
        super().__init__()
        self.convolution = nn.Conv1d(
            in_channels, out_channels, kernel_size=3, padding=1
        )
        self.batch_norm = nn.BatchNorm1d(out_channels) if use_bn else nn.Identity()
        self.act = OfficialCNOLReLU1d(in_size, out_size)

    def forward(self, x):
        return self.act(self.batch_norm(self.convolution(x)))

class OfficialCNOLiftProject1d(nn.Module):
    def __init__(self, in_channels, out_channels, size, latent_dim=64):
        super().__init__()
        self.inter_block = OfficialCNOBlock1d(
            in_channels, latent_dim, size, size, use_bn=False
        )
        self.convolution = nn.Conv1d(
            latent_dim, out_channels, kernel_size=3, padding=1
        )

    def forward(self, x):
        return self.convolution(self.inter_block(x))

class OfficialCNOResidualBlock1d(nn.Module):
    def __init__(self, channels, size, use_bn=True):
        super().__init__()
        self.convolution1 = nn.Conv1d(
            channels, channels, kernel_size=3, padding=1
        )
        self.convolution2 = nn.Conv1d(
            channels, channels, kernel_size=3, padding=1
        )
        self.batch_norm1 = nn.BatchNorm1d(channels) if use_bn else nn.Identity()
        self.batch_norm2 = nn.BatchNorm1d(channels) if use_bn else nn.Identity()
        self.act = OfficialCNOLReLU1d(size, size)

    def forward(self, x):
        out = self.batch_norm1(self.convolution1(x))
        out = self.act(out)
        out = self.batch_norm2(self.convolution2(out))
        return x + out

class OfficialCNOResNet1d(nn.Module):
    def __init__(self, channels, size, num_blocks, use_bn=True):
        super().__init__()
        self.blocks = nn.Sequential(*[
            OfficialCNOResidualBlock1d(channels, size, use_bn)
            for _ in range(int(num_blocks))
        ])

    def forward(self, x):
        return self.blocks(x)

class OfficialCNO1d(nn.Module):
    """Official simplified CNO1d architecture with task-specific channel counts."""

    def __init__(self, in_dim, out_dim, size, n_layers=4, n_res=4,
                 n_res_neck=4, channel_multiplier=16, use_bn=False):
        super().__init__()
        self.n_layers = int(n_layers)
        self.lift_dim = int(channel_multiplier) // 2

        encoder_features = [self.lift_dim]
        for i in range(self.n_layers):
            encoder_features.append((2 ** i) * int(channel_multiplier))
        decoder_features_in = list(reversed(encoder_features[1:]))
        decoder_features_out = list(reversed(encoder_features[:-1]))
        for i in range(1, self.n_layers):
            decoder_features_in[i] *= 2

        encoder_sizes = [int(size) // (2 ** i) for i in range(self.n_layers + 1)]
        decoder_sizes = [
            int(size) // (2 ** (self.n_layers - i))
            for i in range(self.n_layers + 1)
        ]
        if min(encoder_sizes) < 1:
            raise ValueError(
                f"CNO has too many levels ({self.n_layers}) for spatial size {size}."
            )

        self.lift = OfficialCNOLiftProject1d(
            in_dim, encoder_features[0], int(size)
        )
        self.project = OfficialCNOLiftProject1d(
            encoder_features[0] + decoder_features_out[-1],
            out_dim,
            int(size),
        )
        self.encoder = nn.ModuleList([
            OfficialCNOBlock1d(
                encoder_features[i],
                encoder_features[i + 1],
                encoder_sizes[i],
                encoder_sizes[i + 1],
                use_bn,
            )
            for i in range(self.n_layers)
        ])
        self.ed_expansion = nn.ModuleList([
            OfficialCNOBlock1d(
                encoder_features[i],
                encoder_features[i],
                encoder_sizes[i],
                decoder_sizes[self.n_layers - i],
                use_bn,
            )
            for i in range(self.n_layers + 1)
        ])
        self.decoder = nn.ModuleList([
            OfficialCNOBlock1d(
                decoder_features_in[i],
                decoder_features_out[i],
                decoder_sizes[i],
                decoder_sizes[i + 1],
                use_bn,
            )
            for i in range(self.n_layers)
        ])
        self.res_nets = nn.ModuleList([
            OfficialCNOResNet1d(
                encoder_features[level],
                encoder_sizes[level],
                n_res,
                use_bn,
            )
            for level in range(self.n_layers)
        ])
        self.res_net_neck = OfficialCNOResNet1d(
            encoder_features[self.n_layers],
            encoder_sizes[self.n_layers],
            n_res_neck,
            use_bn,
        )

    def forward(self, x):
        x = self.lift(x)
        skip = []
        for i in range(self.n_layers):
            skip.append(self.res_nets[i](x))
            x = self.encoder[i](x)
        x = self.res_net_neck(x)
        for i in range(self.n_layers):
            if i == 0:
                x = self.ed_expansion[self.n_layers](x)
            else:
                x = torch.cat(
                    [x, self.ed_expansion[self.n_layers - i](skip[-i])],
                    dim=1,
                )
            x = self.decoder[i](x)
        x = torch.cat([x, self.ed_expansion[0](skip[0])], dim=1)
        return self.project(x)

class PeriodicConv1d(nn.Module):
    """Conv1d with explicit circular padding for the periodic Burgers domain."""

    def __init__(self, in_channels, out_channels, kernel_size=3, bias=False):
        super().__init__()
        if int(kernel_size) % 2 != 1:
            raise ValueError("PeriodicConv1d expects an odd kernel size.")
        self.pad = int(kernel_size) // 2
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=int(kernel_size),
            padding=0,
            bias=bias,
        )

    def forward(self, x):
        if self.pad:
            x = F.pad(x, (self.pad, self.pad), mode="circular")
        return self.conv(x)

class PeriodicPDEBenchUNet1d(nn.Module):
    """
    PDEBench UNet1d topology with periodic rather than zero padding.

    The encoder/decoder widths, max pooling, transposed convolutions,
    BatchNorm1d layers, and tanh nonlinearities match the upstream backbone.
    Circular padding is the only architectural boundary-condition adaptation.
    """

    def __init__(self, in_channels=1, out_channels=1, init_features=32):
        super().__init__()
        f = int(init_features)
        self.encoder1 = self._block(in_channels, f)
        self.pool1 = nn.MaxPool1d(2, 2)
        self.encoder2 = self._block(f, 2 * f)
        self.pool2 = nn.MaxPool1d(2, 2)
        self.encoder3 = self._block(2 * f, 4 * f)
        self.pool3 = nn.MaxPool1d(2, 2)
        self.encoder4 = self._block(4 * f, 8 * f)
        self.pool4 = nn.MaxPool1d(2, 2)
        self.bottleneck = self._block(8 * f, 16 * f)
        self.upconv4 = nn.ConvTranspose1d(16 * f, 8 * f, 2, 2)
        self.decoder4 = self._block(16 * f, 8 * f)
        self.upconv3 = nn.ConvTranspose1d(8 * f, 4 * f, 2, 2)
        self.decoder3 = self._block(8 * f, 4 * f)
        self.upconv2 = nn.ConvTranspose1d(4 * f, 2 * f, 2, 2)
        self.decoder2 = self._block(4 * f, 2 * f)
        self.upconv1 = nn.ConvTranspose1d(2 * f, f, 2, 2)
        self.decoder1 = self._block(2 * f, f)
        self.conv = nn.Conv1d(f, out_channels, kernel_size=1)

    @staticmethod
    def _block(in_channels, out_channels):
        return nn.Sequential(OrderedDict([
            ("conv1", PeriodicConv1d(in_channels, out_channels, 3, False)),
            ("norm1", nn.BatchNorm1d(out_channels)),
            ("tanh1", nn.Tanh()),
            ("conv2", PeriodicConv1d(out_channels, out_channels, 3, False)),
            ("norm2", nn.BatchNorm1d(out_channels)),
            ("tanh2", nn.Tanh()),
        ]))

    @staticmethod
    def _match_length(x, skip):
        diff = skip.size(-1) - x.size(-1)
        if diff != 0:
            x = F.pad(x, [diff // 2, diff - diff // 2], mode="circular")
        return x

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))
        bottleneck = self.bottleneck(self.pool4(enc4))
        dec4 = self._match_length(self.upconv4(bottleneck), enc4)
        dec4 = self.decoder4(torch.cat((dec4, enc4), dim=1))
        dec3 = self._match_length(self.upconv3(dec4), enc3)
        dec3 = self.decoder3(torch.cat((dec3, enc3), dim=1))
        dec2 = self._match_length(self.upconv2(dec3), enc2)
        dec2 = self.decoder2(torch.cat((dec2, enc2), dim=1))
        dec1 = self._match_length(self.upconv1(dec2), enc1)
        dec1 = self.decoder1(torch.cat((dec1, enc1), dim=1))
        return self.conv(dec1)

class PaperUNetAR1d(nn.Module):
    """One-state, one-step periodic U-Net used only through recursive rollout."""

    def __init__(self):
        super().__init__()
        self.unet = PeriodicPDEBenchUNet1d(
            in_channels=1,
            out_channels=1,
            init_features=UNET_AR_FEATURES,
        )

    def forward(self, u_norm):
        return self.unet(u_norm.unsqueeze(1))[:, 0]

class PaperFFNO1d(nn.Module):
    """
    Competitive one-axis adaptation of FourierFlow's FNOFactorized2DBlock.

    Burgers has one spatial axis, hence each layer contains exactly one
    factorized 1D Fourier transform. The layer widths, depth, shared Fourier
    weights, feed-forward blocks, weight normalization, and initialization
    follow the FourierFlow turbulent Markov configuration. The state is given
    the same periodic coordinate features used by the AWFNO benchmark.
    """

    def __init__(self, x_grid):
        super().__init__()
        x = torch.tensor(x_grid, dtype=torch.float32)
        xn = (x - x.min()) / (x.max() - x.min() + 1.0e-12)
        grid = torch.stack(
            [
                xn,
                torch.sin(2.0 * np.pi * xn),
                torch.cos(2.0 * np.pi * xn),
            ],
            dim=-1,
        )
        self.register_buffer(
            "grid",
            grid.unsqueeze(0),
            persistent=False,
        )
        self.in_proj = FFNOWLinear(
            BASELINE_INPUT_CHANNELS,
            FFNO_WIDTH,
            use_weight_norm=FFNO_WEIGHT_NORM,
        )
        shared_weight = None
        if FFNO_SHARE_FOURIER_WEIGHT:
            weight = torch.empty(FFNO_WIDTH, FFNO_WIDTH, FFNO_MODES, 2)
            nn.init.xavier_normal_(weight, gain=0.1)
            shared_weight = nn.Parameter(weight)
            self.shared_fourier_weight = shared_weight
        else:
            self.register_parameter("shared_fourier_weight", None)
        self.spectral_layers = nn.ModuleList([
            FFNOSpectralLayer1d(
                FFNO_WIDTH,
                FFNO_MODES,
                fourier_weight=shared_weight,
                factor=FFNO_FACTOR,
                n_ff_layers=FFNO_FF_LAYERS,
                use_weight_norm=FFNO_WEIGHT_NORM,
            )
            for _ in range(FFNO_LAYERS)
        ])
        self.out = nn.Sequential(
            FFNOWLinear(FFNO_WIDTH, 128, use_weight_norm=FFNO_WEIGHT_NORM),
            FFNOWLinear(128, 1, use_weight_norm=FFNO_WEIGHT_NORM),
        )

    def forward(self, u_norm, noise_std=0.0):
        batch = u_norm.shape[0]
        if self.training and float(noise_std) > 0.0:
            u_norm = u_norm + float(noise_std) * torch.randn_like(u_norm)
        features = torch.cat(
            [u_norm.unsqueeze(-1), self.grid.expand(batch, -1, -1)],
            dim=-1,
        )
        x = self.in_proj(features)
        backcast = None
        for layer in self.spectral_layers:
            backcast = layer(x)
            x = x + backcast
        return self.out(backcast).squeeze(-1).contiguous()

class PaperCNODirect1d(nn.Module):
    """
    Upstream CNO1d with two input channels (u0, x) and Nt-1 output channels.

    It is not autoregressive: one model call returns all future snapshots.
    """

    def __init__(self, nt, x_grid):
        super().__init__()
        self.nt = int(nt)
        x = torch.tensor(x_grid, dtype=torch.float32)
        xn = (x - x.min()) / (x.max() - x.min() + 1.0e-12)
        self.register_buffer("x_feature", xn.view(1, 1, -1), persistent=False)
        self.cno = OfficialCNO1d(
            in_dim=2,
            out_dim=self.nt - 1,
            size=len(x_grid),
            n_layers=CNO_LAYERS,
            n_res=CNO_RES_BLOCKS,
            n_res_neck=CNO_NECK_BLOCKS,
            channel_multiplier=CNO_CHANNEL_MULTIPLIER,
            use_bn=CNO_USE_BATCH_NORM,
        )

    def forward(self, u0):
        x = torch.cat(
            [u0.unsqueeze(1), self.x_feature.expand(u0.shape[0], -1, -1)],
            dim=1,
        )
        return self.cno(x).permute(0, 2, 1).contiguous()

def relative_lp_loss(pred, target, p=2, eps=1.0e-8):
    """Mean per-sample relative Lp loss used by the upstream operator codes."""
    batch = pred.shape[0]
    diff = torch.linalg.vector_norm(
        (pred - target).reshape(batch, -1), ord=p, dim=1
    )
    scale = torch.linalg.vector_norm(
        target.reshape(batch, -1), ord=p, dim=1
    ).clamp_min(eps)
    return torch.mean(diff / scale)

def make_paper_ar_predict(model, state_normalizer, kind):
    kind = str(kind).lower()

    def predict(u_state, training_noise=False):
        u_norm = state_normalizer.encode(u_state)
        if kind == "ffno":
            next_norm = model(
                u_norm,
                noise_std=FFNO_NOISE_STD if training_noise else 0.0,
            )
        elif kind == "unet":
            next_norm = model(u_norm)
        else:
            raise ValueError(f"Unknown autoregressive model kind: {kind}")
        return state_normalizer.decode(next_norm)

    return predict

def train_epoch_paper_ar(
    model,
    loader,
    optimizer,
    predict,
    loss_kind,
    gradient_clip,
    batch_scheduler=None,
):
    model.train()
    total_loss = 0.0
    total_l2 = 0.0
    n = 0
    for batch in loader:
        u_start = to_device(batch["u_start"])
        u_next = to_device(batch["u_next"])
        optimizer.zero_grad(set_to_none=True)
        pred = predict(u_start, training_noise=True)
        if loss_kind == "relative_l2":
            loss = relative_lp_loss(pred, u_next, p=2)
        elif loss_kind == "mse":
            loss = F.mse_loss(pred, u_next)
        else:
            raise ValueError(loss_kind)
        loss.backward()
        if gradient_clip > 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), gradient_clip
            )
        optimizer.step()
        if batch_scheduler is not None:
            batch_scheduler.step()
        bs = u_start.shape[0]
        total_loss += loss.item() * bs
        total_l2 += (
            abs_l2_field_batch(pred.detach(), u_next).mean().item() * bs
        )
        n += bs
    return {
        "loss": total_loss / max(n, 1),
        "abs_l2": total_l2 / max(n, 1),
    }

@torch.no_grad()
def eval_epoch_paper_ar(model, loader, predict, loss_kind):
    model.eval()
    total_loss = 0.0
    total_l2 = 0.0
    n = 0
    for batch in loader:
        u_start = to_device(batch["u_start"])
        u_next = to_device(batch["u_next"])
        pred = predict(u_start, training_noise=False)
        if loss_kind == "relative_l2":
            loss = relative_lp_loss(pred, u_next, p=2)
        elif loss_kind == "mse":
            loss = F.mse_loss(pred, u_next)
        else:
            raise ValueError(loss_kind)
        bs = u_start.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_field_batch(pred, u_next).mean().item() * bs
        n += bs
    return {
        "loss": total_loss / max(n, 1),
        "abs_l2": total_l2 / max(n, 1),
    }

def fit_paper_ar_model(
    model,
    train_loader,
    val_pair_loader,
    U_val,
    predict,
    name,
    loss_kind,
    learning_rate,
    weight_decay,
    val_pair_interval,
    val_rollout_interval,
):
    """
    Train a one-step map and select its checkpoint using recursive validation.

    No exact intermediate validation or test state is ever supplied during a
    rollout. Pair validation is diagnostic only; model selection uses the
    absolute full-trajectory L2 on validation trajectories.
    """
    if name == "U-Net-AR":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=max(EPOCHS // 5, 1),
            gamma=0.5,
        )
        batch_scheduler = None
    else:
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
        # Reproduce the optimizer protocol of the earlier competitive Burgers
        # F-FNO run. The upstream absolute 500/100000-step schedule was tuned
        # for a run with about 100000 updates and is not portable to this
        # sample-size sweep.
        print(
            f"[F-FNO] optimizer schedule: epoch cosine over {EPOCHS} epochs | "
            f"{EPOCHS * len(train_loader)} optimizer updates"
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS,
        )
        batch_scheduler = None

    best_val = float("inf")
    best_state = None
    best_epoch = None
    last_pair_loss = float("nan")
    last_pair_l2 = float("nan")
    last_rollout_l2 = float("nan")
    last_final_l2 = float("nan")
    history = {
        "train_loss": [],
        "train_l2": [],
        "val_loss": [],
        "val_l2": [],
        "val_final_l2": [],
        "learning_rate": [],
    }
    start_time = time.time()
    pair_interval = max(int(val_pair_interval), 1)
    rollout_interval = max(int(val_rollout_interval), 1)

    for epoch in range(1, EPOCHS + 1):
        train_stats = train_epoch_paper_ar(
            model,
            train_loader,
            optimizer,
            predict,
            loss_kind,
            (
                UNET_AR_GRAD_CLIP
                if name == "U-Net-AR"
                else FFNO_GRAD_CLIP
            ),
            batch_scheduler=batch_scheduler,
        )
        if epoch == 1 or epoch == EPOCHS or epoch % pair_interval == 0:
            pair_stats = eval_epoch_paper_ar(
                model, val_pair_loader, predict, loss_kind
            )
            last_pair_loss = pair_stats["loss"]
            last_pair_l2 = pair_stats["abs_l2"]

        if epoch == EPOCHS or epoch % rollout_interval == 0:
            val_rollout = autoregressive_rollout_generic(
                model,
                U_val[:, :, 0],
                U_val.shape[-1],
                lambda u: predict(u, training_noise=False),
                name=f"{name} validation",
                batch_size=(
                    UNET_AR_BATCH_SIZE
                    if name == "U-Net-AR"
                    else FFNO_BATCH_SIZE
                ),
                print_diagnostics=False,
            )
            rollout_stats = rollout_l2_metrics(val_rollout, U_val)
            last_rollout_l2 = rollout_stats["full_l2_mean"]
            last_final_l2 = rollout_stats["final_l2_mean"]
            if np.isfinite(last_rollout_l2) and last_rollout_l2 < best_val:
                best_val = last_rollout_l2
                best_state = clean_state_dict(model.state_dict())
                best_epoch = int(epoch)

        if scheduler is not None:
            scheduler.step()
        history["train_loss"].append(train_stats["loss"])
        history["train_l2"].append(train_stats["abs_l2"])
        history["val_loss"].append(last_pair_loss)
        history["val_l2"].append(last_rollout_l2)
        history["val_final_l2"].append(last_final_l2)
        history["learning_rate"].append(
            float(optimizer.param_groups[0]["lr"])
        )

        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            print(
                f"[{name}] {epoch:04d}/{EPOCHS} | "
                f"train loss {train_stats['loss']:.6e} | "
                f"train one-step L2 {train_stats['abs_l2']:.6e} | "
                f"val pair loss {last_pair_loss:.6e} | "
                f"val one-step L2 {last_pair_l2:.6e} | "
                f"val recursive full L2 {last_rollout_l2:.6e} | "
                f"lr {optimizer.param_groups[0]['lr']:.3e}"
            )

    if best_state is None:
        raise RuntimeError(
            f"{name} never produced a finite full validation rollout. "
            "This run is invalid; do not report its test metrics."
        )
    model.load_state_dict(best_state, strict=True)
    print(
        f"[{name}] selected epoch {best_epoch} with validation full L2 "
        f"{best_val:.6e}"
    )
    return {
        "model": model,
        "best_val_l2": best_val,
        "best_epoch": best_epoch,
        "train_time_sec": time.time() - start_time,
        "history": history,
    }

def train_epoch_cno_direct(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_l2 = 0.0
    n = 0
    for batch in loader:
        u0 = to_device(batch["u0"])
        future = to_device(batch["U_future"])
        optimizer.zero_grad(set_to_none=True)
        pred = model(u0)
        # The upstream CNO1d training demo uses ordinary mean L1 loss.
        loss = F.l1_loss(pred, future)
        loss.backward()
        if CNO_DIRECT_GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), CNO_DIRECT_GRAD_CLIP
            )
        optimizer.step()
        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(
            torch.cat([u0.unsqueeze(-1), pred.detach()], dim=-1),
            torch.cat([u0.unsqueeze(-1), future], dim=-1),
        ).mean().item() * bs
        n += bs
    return {
        "loss": total_loss / max(n, 1),
        "abs_l2": total_l2 / max(n, 1),
    }

@torch.no_grad()
def eval_epoch_cno_direct(model, loader):
    model.eval()
    total_loss = 0.0
    total_l2 = 0.0
    total_final = 0.0
    n = 0
    for batch in loader:
        u0 = to_device(batch["u0"])
        future = to_device(batch["U_future"])
        pred = model(u0)
        full_pred = torch.cat([u0.unsqueeze(-1), pred], dim=-1)
        full_true = torch.cat([u0.unsqueeze(-1), future], dim=-1)
        loss = F.l1_loss(pred, future)
        bs = u0.shape[0]
        total_loss += loss.item() * bs
        total_l2 += abs_l2_window_batch(
            full_pred, full_true
        ).mean().item() * bs
        total_final += abs_l2_field_batch(
            pred[:, :, -1], future[:, :, -1]
        ).mean().item() * bs
        n += bs
    return {
        "loss": total_loss / max(n, 1),
        "abs_l2": total_l2 / max(n, 1),
        "final_l2": total_final / max(n, 1),
    }

def fit_cno_direct_model(model, train_loader, val_loader):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CNO_DIRECT_LR,
        weight_decay=CNO_DIRECT_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=15,
        gamma=0.5,
    )
    best_val = float("inf")
    best_state = None
    last_val = {
        "loss": float("nan"),
        "abs_l2": float("nan"),
        "final_l2": float("nan"),
    }
    history = {
        "train_loss": [],
        "train_l2": [],
        "val_loss": [],
        "val_l2": [],
        "val_final_l2": [],
    }
    start_time = time.time()
    interval = max(int(CNO_DIRECT_VAL_INTERVAL), 1)
    for epoch in range(1, EPOCHS + 1):
        train_stats = train_epoch_cno_direct(model, train_loader, optimizer)
        if epoch == 1 or epoch == EPOCHS or epoch % interval == 0:
            last_val = eval_epoch_cno_direct(model, val_loader)
            if last_val["abs_l2"] < best_val:
                best_val = last_val["abs_l2"]
                best_state = clean_state_dict(model.state_dict())
        scheduler.step()
        history["train_loss"].append(train_stats["loss"])
        history["train_l2"].append(train_stats["abs_l2"])
        history["val_loss"].append(last_val["loss"])
        history["val_l2"].append(last_val["abs_l2"])
        history["val_final_l2"].append(last_val["final_l2"])
        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            print(
                f"[CNO-Direct] {epoch:04d}/{EPOCHS} | "
                f"train L1 {train_stats['loss']:.6e} | "
                f"train full L2 {train_stats['abs_l2']:.6e} | "
                f"val L1 {last_val['loss']:.6e} | "
                f"val full L2 {last_val['abs_l2']:.6e} | "
                f"val final L2 {last_val['final_l2']:.6e}"
            )
    if best_state is None:
        raise RuntimeError("CNO-Direct never completed validation.")
    model.load_state_dict(best_state, strict=True)
    return {
        "model": model,
        "best_val_l2": best_val,
        "train_time_sec": time.time() - start_time,
        "history": history,
    }

@torch.no_grad()
def autoregressive_rollout_generic(
    model,
    u0_test_np,
    nt,
    predict,
    name,
    batch_size=None,
    print_diagnostics=True,
):
    """One-step recursive rollout for predictors with predict(u) signature."""
    model.eval()
    batch_size = batch_size or BATCH_SIZE
    n_samples, nx = u0_test_np.shape
    U_roll = np.zeros((n_samples, nx, nt), dtype=np.float32)
    U_roll[:, :, 0] = np.asarray(u0_test_np, dtype=np.float32)
    current = np.asarray(u0_test_np, dtype=np.float32)
    if print_diagnostics:
        print(f"\nOne-step autoregressive rollout: {name}")
    for step in range(nt - 1):
        next_parts = []
        for i0 in range(0, n_samples, batch_size):
            i1 = min(i0 + batch_size, n_samples)
            state = torch.tensor(
                current[i0:i1], dtype=torch.float32, device=device
            )
            next_parts.append(predict(state).detach().cpu().numpy())
        current = np.concatenate(next_parts, axis=0).astype(
            np.float32, copy=False
        )
        U_roll[:, :, step + 1] = current
        if print_diagnostics and (
            step == 0 or (step + 1) % 10 == 0 or step == nt - 2
        ):
            print(f"{name} step {step + 1:04d}/{nt - 1:04d}")
    return U_roll

@torch.no_grad()
def direct_full_trajectory_prediction(model, u0_test_np, nt, predict, name="FNO", batch_size=None):
    model.eval()
    batch_size = batch_size or BATCH_SIZE
    N, nx = u0_test_np.shape
    U_pred = np.zeros((N, nx, nt), dtype=np.float32)
    U_pred[:, :, 0] = u0_test_np
    print(f"\nDirect full-trajectory prediction: {name}")
    for i0 in range(0, N, batch_size):
        i1 = min(i0 + batch_size, N)
        u0 = torch.tensor(u0_test_np[i0:i1], dtype=torch.float32, device=device)
        Uf_pred = predict(u0)
        U_pred[i0:i1, :, 1:] = Uf_pred.detach().cpu().numpy()
    return U_pred

def print_terminal_errors(U_roll, U_true, name):
    terminal_l2 = l2_per_timestep_np(U_roll, U_true)[:, -1]
    print(f"{name} final-time L2: mean={terminal_l2.mean():.6e}, median={np.median(terminal_l2):.6e}, max={terminal_l2.max():.6e}")

def get_plot_time_indices(nt):
    out = []
    for item in PLOT_TIME_INDICES_MANUAL:
        out.append(nt - 1 if item == "final" else int(item))
    clean = []
    for k in out:
        if 0 <= k < nt and k not in clean:
            clean.append(k)
    return clean or [0, nt // 2, nt - 1]

def plot_training_curves(hist, name):
    if not SAVE_PLOTS or not MAKE_TRAINING_CURVE_PLOTS:
        return
    plt.figure(figsize=(8, 5))
    plt.semilogy(hist["train_loss"], label="train loss")
    plt.semilogy(hist["val_loss"], label="val loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = PLOT_DIR / f"{safe_name(name)}_loss_curves_{BASE_TAG}.png"
    plt.savefig(path, dpi=PLOT_DPI)
    plt.close()
    print("Saved:", path)

    plt.figure(figsize=(8, 5))
    plt.semilogy(hist["train_l2"], label="train L2")
    plt.semilogy(hist["val_l2"], label="val L2")
    plt.semilogy(hist["val_final_l2"], label="val final L2")
    plt.xlabel("epoch")
    plt.ylabel("physical L2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = PLOT_DIR / f"{safe_name(name)}_l2_curves_{BASE_TAG}.png"
    plt.savefig(path, dpi=PLOT_DPI)
    plt.close()
    print("Saved:", path)

def plot_per_test_l2_curves(l2_dict):
    if not SAVE_PLOTS or not MAKE_PER_TEST_L2_PLOTS:
        return
    out_dir = PLOT_DIR / "l2_error_per_timestep"
    out_dir.mkdir(parents=True, exist_ok=True)
    n_available = next(iter(l2_dict.values())).shape[0]
    n_plot = n_available if MAX_TEST_CASES_TO_PLOT is None else min(MAX_TEST_CASES_TO_PLOT, n_available)
    for i in range(n_plot):
        plt.figure(figsize=(8, 5))
        for name, l2_t in l2_dict.items():
            plt.plot(t_grid, l2_t[i], label=name, linewidth=2)
        plt.xlabel("time")
        plt.ylabel(r"$E_{L^2}$")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        path = out_dir / f"test_{i:03d}_L2_error_per_timestep.png"
        plt.savefig(path, dpi=PLOT_DPI)
        plt.close()
        print("Saved:", path)

def plot_solution_curves(U_true, rollouts):
    if not SAVE_PLOTS or not MAKE_SOLUTION_PLOTS:
        return
    out_dir = PLOT_DIR / "selected_test_case_solutions"
    out_dir.mkdir(parents=True, exist_ok=True)
    time_ids = get_plot_time_indices(U_true.shape[-1])
    n_available = U_true.shape[0]
    n_plot = n_available if MAX_TEST_CASES_TO_PLOT is None else min(MAX_TEST_CASES_TO_PLOT, n_available)
    for i in range(n_plot):
        fig, axes = plt.subplots(len(time_ids), 1, figsize=(9, 3 * len(time_ids)), squeeze=False, constrained_layout=True)
        for row, tidx in enumerate(time_ids):
            ax = axes[row, 0]
            ax.plot(x_grid_np, U_true[i, :, tidx], label="true", linewidth=2.4)
            for name, arr in rollouts.items():
                ax.plot(x_grid_np, arr[i, :, tidx], label=name, linewidth=1.7)
            ax.set_title(f"test {i}, time index {tidx}")
            ax.set_xlabel("x")
            ax.set_ylabel("u")
            ax.grid(True, alpha=0.3)
            ax.legend()
        path = out_dir / f"test_{i:03d}_solution_curves.png"
        plt.savefig(path, dpi=PLOT_DPI)
        plt.close(fig)
        print("Saved:", path)

def _load_or_fit_baseline(
    path,
    model,
    fit_function,
    checkpoint_meta,
    legacy_paths=(),
):
    if path.exists() and not FORCE_RETRAIN:
        print("\nLoading cached model:", path)
        meta = load_model_checkpoint(path, model)
        return {
            "model": model,
            "best_val_l2": meta.get("best_val_l2", float("nan")),
            "best_epoch": meta.get("best_epoch", None),
            "train_time_sec": meta.get("train_time_sec", 0.0),
            "history": meta.get("history", None),
            "selected_protocol": meta.get("selected_protocol", None),
        }

    if not FORCE_RETRAIN:
        existing_legacy = [Path(item) for item in legacy_paths if Path(item).exists()]
        if len(existing_legacy) == 1:
            legacy_path = existing_legacy[0]
            print("\nMigrating compatible v1 checkpoint:", legacy_path)
            legacy_meta = load_model_checkpoint(legacy_path, model)
            migrated_meta = dict(checkpoint_meta)
            migrated_meta.update({
                "best_val_l2": legacy_meta.get("best_val_l2", float("nan")),
                "best_epoch": legacy_meta.get("best_epoch", None),
                "train_time_sec": legacy_meta.get("train_time_sec", 0.0),
                "history": legacy_meta.get("history", None),
                "parameter_count": count_params(model),
                "migrated_from": str(legacy_path),
            })
            if "selected_protocol" in legacy_meta:
                migrated_meta["selected_protocol"] = legacy_meta[
                    "selected_protocol"
                ]
            save_model_checkpoint(path, model, migrated_meta)
            return {
                "model": model,
                "best_val_l2": migrated_meta["best_val_l2"],
                "best_epoch": migrated_meta["best_epoch"],
                "train_time_sec": migrated_meta["train_time_sec"],
                "history": migrated_meta["history"],
                "selected_protocol": migrated_meta.get(
                    "selected_protocol", None
                ),
            }
        if len(existing_legacy) > 1:
            print(
                "\nMultiple legacy checkpoints match; none was reused:",
                [str(item) for item in existing_legacy],
            )

    fit = fit_function()
    metadata = dict(checkpoint_meta)
    metadata.update({
        "best_val_l2": fit["best_val_l2"],
        "best_epoch": fit.get("best_epoch", None),
        "train_time_sec": fit["train_time_sec"],
        "history": fit["history"],
        "parameter_count": count_params(model),
    })
    if "selected_protocol" in fit:
        metadata["selected_protocol"] = fit["selected_protocol"]
    save_model_checkpoint(path, fit["model"], metadata)
    return fit

def paper_v1_checkpoint_candidates(n_train, model_stem):
    """
    Locate completed v1 U-Net/CNO checkpoints for migration.

    F-FNO is intentionally excluded because the paper-v1 batch-512,
    relative-loss protocol is incompatible with the competitive protocol
    restored in paper-v3.
    """
    if not REUSE_V1_NON_FFNO:
        return ()
    return tuple(
        sorted(
            PAPER_V1_MODEL_DIR.glob(
                f"N{int(n_train)}_{str(model_stem)}_*.pt"
            )
        )
    )

def competitive_ffno_checkpoint_candidates(n_train):
    """
    Find checkpoints from the earlier competitive F-FNO notebook.

    That run used the same data split, one-step adjacent-pair training,
    recursive rollout, periodic coordinate features, MSE, batch size 16, and
    epoch cosine schedule restored by paper protocol v3.
    """
    if not REUSE_COMPETITIVE_FFNO:
        return ()
    old_output = OUTPUT_ROOT.parent / "burgers_official_baselines"
    if not old_output.exists():
        return ()
    return tuple(
        sorted(
            old_output.glob(
                "cache_*/models/"
                f"N{int(n_train)}_FFNO_markov_*.pt"
            )
        )
    )

def _synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()

def _timed_prediction(function):
    _synchronize_device()
    start = time.perf_counter()
    value = function()
    _synchronize_device()
    return value, time.perf_counter() - start

def make_paper_baseline_tag(n_train):
    data_signature = {
        "data_path_name": DATA_PATH.name,
        "data_path_exists": DATA_PATH.exists(),
        "data_path_size": DATA_PATH.stat().st_size if DATA_PATH.exists() else None,
    }
    return short_hash({
        "suite": "paper_baselines",
        "protocol_version": PAPER_BASELINE_VERSION,
        "seed": SEED,
        "split_mode": SPLIT_MODE,
        "split_seed": SPLIT_SEED,
        "data_signature": data_signature,
        "all_sample_sizes": ALL_SAMPLE_SIZES,
        "N_train": int(n_train),
        "N_val": N_VAL,
        "N_test": N_TEST,
        "Nx": Nx,
        "Nt": Nt,
        "epochs": EPOCHS,
        "metrics": {
            "use_trapezoidal_time_l2": USE_TRAPEZOIDAL_TIME_L2,
            "include_t0_in_full_l2": INCLUDE_T0_IN_FULL_L2,
            "dx": DX,
            "dt_snapshot": DT_SNAPSHOT,
        },
        "unet_ar": {
            "features": UNET_AR_FEATURES,
            "batch_size": UNET_AR_BATCH_SIZE,
            "lr": UNET_AR_LR,
            "weight_decay": UNET_AR_WEIGHT_DECAY,
            "val_pair_interval": UNET_AR_VAL_PAIR_INTERVAL,
            "val_rollout_interval": UNET_AR_VAL_ROLLOUT_INTERVAL,
            "grad_clip": UNET_AR_GRAD_CLIP,
            "loss": "physical-space MSE on all adjacent pairs",
            "padding": "circular",
        },
        "cno_direct": {
            "layers": CNO_LAYERS,
            "res_blocks": CNO_RES_BLOCKS,
            "neck_blocks": CNO_NECK_BLOCKS,
            "channel_multiplier": CNO_CHANNEL_MULTIPLIER,
            "batch_norm": CNO_USE_BATCH_NORM,
            "batch_size": CNO_DIRECT_BATCH_SIZE,
            "lr": CNO_DIRECT_LR,
            "weight_decay": CNO_DIRECT_WEIGHT_DECAY,
            "val_interval": CNO_DIRECT_VAL_INTERVAL,
            "grad_clip": CNO_DIRECT_GRAD_CLIP,
            "loss": "mean L1 on the complete future trajectory",
            "scheduler_step": 15,
            "scheduler_gamma": 0.5,
        },
        "ffno": {
            "width": FFNO_WIDTH,
            "layers": FFNO_LAYERS,
            "modes": FFNO_MODES,
            "factor": FFNO_FACTOR,
            "ff_layers": FFNO_FF_LAYERS,
            "share_fourier_weight": FFNO_SHARE_FOURIER_WEIGHT,
            "weight_norm": FFNO_WEIGHT_NORM,
            "noise_std": FFNO_NOISE_STD,
            "batch_size": FFNO_BATCH_SIZE,
            "lr": FFNO_LR,
            "weight_decay": FFNO_WEIGHT_DECAY,
            "schedule": "CosineAnnealingLR over training epochs",
            "grad_clip": FFNO_GRAD_CLIP,
            "val_pair_interval": FFNO_PAPER_VAL_PAIR_INTERVAL,
            "val_rollout_interval": FFNO_PAPER_VAL_ROLLOUT_INTERVAL,
            "loss": "physical-space MSE on all adjacent pairs",
            "coordinate_features": "x, sin(2*pi*x), cos(2*pi*x)",
        },
    })

def build_paper_baseline_objects(n_train, U_train_pool, U_val):
    U_train = np.asarray(U_train_pool[:n_train], dtype=np.float32)
    U_val = np.asarray(U_val, dtype=np.float32)

    pair_train_ds = MarkovPairDataset1D(U_train)
    pair_val_ds = MarkovPairDataset1D(U_val)
    direct_train_ds = FullTrajectoryDataset1D(U_train)
    direct_val_ds = FullTrajectoryDataset1D(U_val)

    unet_generator = torch.Generator()
    unet_generator.manual_seed(SEED + 1100 + int(n_train))
    ffno_generator = torch.Generator()
    ffno_generator.manual_seed(SEED + 1200 + int(n_train))
    cno_generator = torch.Generator()
    cno_generator.manual_seed(SEED + 1300 + int(n_train))

    unet_train_loader = DataLoader(
        pair_train_ds,
        batch_size=UNET_AR_BATCH_SIZE,
        shuffle=True,
        generator=unet_generator,
        pin_memory=PIN_MEMORY,
    )
    unet_val_loader = DataLoader(
        pair_val_ds,
        batch_size=UNET_AR_BATCH_SIZE,
        shuffle=False,
        pin_memory=PIN_MEMORY,
    )
    ffno_train_loader = DataLoader(
        pair_train_ds,
        batch_size=FFNO_BATCH_SIZE,
        shuffle=True,
        generator=ffno_generator,
        pin_memory=PIN_MEMORY,
    )
    ffno_val_loader = DataLoader(
        pair_val_ds,
        batch_size=FFNO_BATCH_SIZE,
        shuffle=False,
        pin_memory=PIN_MEMORY,
    )
    cno_train_loader = DataLoader(
        direct_train_ds,
        batch_size=CNO_DIRECT_BATCH_SIZE,
        shuffle=True,
        generator=cno_generator,
        pin_memory=PIN_MEMORY,
    )
    cno_val_loader = DataLoader(
        direct_val_ds,
        batch_size=CNO_DIRECT_BATCH_SIZE,
        shuffle=False,
        pin_memory=PIN_MEMORY,
    )
    state_normalizer = GaussianNormalizer(
        torch.as_tensor(U_train, dtype=torch.float32)
    ).to(device)

    print("\nPaper-baseline training data")
    print("----------------------------")
    print("Training trajectories                :", U_train.shape)
    print("Adjacent train/validation pairs      :", len(pair_train_ds), len(pair_val_ds))
    print("Direct CNO train/validation samples  :", len(direct_train_ds), len(direct_val_ds))
    print("U-Net-AR / F-FNO batch sizes         :", UNET_AR_BATCH_SIZE, FFNO_BATCH_SIZE)
    print("CNO-Direct batch size                :", CNO_DIRECT_BATCH_SIZE)
    print("Validation trajectories              :", U_val.shape)
    print("Test data used in training/selection : no")

    return {
        "U_train": U_train,
        "U_val": U_val,
        "unet_train_loader": unet_train_loader,
        "unet_val_loader": unet_val_loader,
        "ffno_train_loader": ffno_train_loader,
        "ffno_val_loader": ffno_val_loader,
        "cno_train_loader": cno_train_loader,
        "cno_val_loader": cno_val_loader,
        "state_normalizer": state_normalizer,
        "n_pair_train": len(pair_train_ds),
        "n_pair_val": len(pair_val_ds),
        "n_direct_train": len(direct_train_ds),
        "n_direct_val": len(direct_val_ds),
    }

def _paper_checkpoint_metadata(n_train, model_name, protocol, adaptation):
    sources = {
        "U-Net-AR": {
            "repository": "https://github.com/pdebench/PDEBench",
            "commit": "4ff3e3a4aa1561721b5571fa3a048a0a463e0568",
            "source_path": "pdebench/models/unet/unet.py",
        },
        "CNO-Direct": {
            "repository": "https://github.com/camlab-ethz/ConvolutionalNeuralOperator",
            "commit": "6e765198aa02b56352e0a3437104b9d9e337176e",
            "source_path": "CNO1d_simplified/CNO1d.py",
        },
        "F-FNO": {
            "repository": "https://github.com/alasdairtran/fourierflow",
            "commit": "82d1689e02c95f7656a7eaa1d88d0ab407b2483d",
            "source_path": "fourierflow/modules/factorized_fno/grid_2d.py",
        },
    }
    return {
        "suite": "paper_baselines",
        "protocol_version": PAPER_BASELINE_VERSION,
        "model_name": model_name,
        "n_train": int(n_train),
        "base_tag": BASE_TAG,
        "protocol": protocol,
        "adaptation": adaptation,
        "source": sources[model_name],
        "checkpoint_selection": (
            "minimum validation absolute full-trajectory L2; test set unused"
        ),
    }

def train_paper_baselines_for_sample_size(n_train, U_train_pool, U_val):
    global BASE_TAG
    BASE_TAG = make_paper_baseline_tag(n_train)
    run_seed = reset_seed_for_sample_size(n_train)
    print("\n" + "=" * 88)
    print(f"PAPER BASELINES | N_train = {n_train}")
    print("=" * 88)
    print("BASE_TAG:", BASE_TAG)
    print("RUN SEED:", run_seed)

    objects = build_paper_baseline_objects(n_train, U_train_pool, U_val)

    unet_model = PaperUNetAR1d().to(device)
    cno_model = PaperCNODirect1d(Nt, x_grid_np).to(device)
    ffno_model = PaperFFNO1d(x_grid_np).to(device)

    unet_predict = make_paper_ar_predict(
        unet_model, objects["state_normalizer"], kind="unet"
    )
    ffno_predict = make_paper_ar_predict(
        ffno_model, objects["state_normalizer"], kind="ffno"
    )

    print("\nArchitectures and inference protocols")
    print("-------------------------------------")
    print(f"U-Net-AR parameters  : {count_params(unet_model):,}")
    print(f"CNO-Direct parameters: {count_params(cno_model):,}")
    print(f"F-FNO parameters     : {count_params(ffno_model):,}")
    print("U-Net-AR : u_n -> u_{n+1}; recursively applied Nt-1 times.")
    print("CNO-Direct: (u_0,x) -> (u_1,...,u_{Nt-1}) in one model call.")
    print("F-FNO    : (u_n,x) -> u_{n+1}; recursively applied Nt-1 times.")
    print("Only u_0 is supplied at test time for every model.")

    unet_path = MODEL_DIR / f"N{n_train}_UNet_AR_{BASE_TAG}.pt"
    cno_path = MODEL_DIR / f"N{n_train}_CNO_Direct_{BASE_TAG}.pt"
    ffno_path = MODEL_DIR / f"N{n_train}_FFNO_AR_{BASE_TAG}.pt"

    unet_fit = _load_or_fit_baseline(
        unet_path,
        unet_model,
        lambda: fit_paper_ar_model(
            unet_model,
            objects["unet_train_loader"],
            objects["unet_val_loader"],
            objects["U_val"],
            unet_predict,
            name="U-Net-AR",
            loss_kind="mse",
            learning_rate=UNET_AR_LR,
            weight_decay=UNET_AR_WEIGHT_DECAY,
            val_pair_interval=UNET_AR_VAL_PAIR_INTERVAL,
            val_rollout_interval=UNET_AR_VAL_ROLLOUT_INTERVAL,
        ),
        _paper_checkpoint_metadata(
            n_train,
            "U-Net-AR",
            (
                "all adjacent training pairs; one-step map; recursive validation "
                "and recursive test rollout from u0"
            ),
            (
                "PDEBench UNet1d topology; initial_step=1; circular convolution "
                "padding for the periodic Burgers domain"
            ),
        ),
        legacy_paths=paper_v1_checkpoint_candidates(n_train, "UNet_AR"),
    )

    cno_fit = _load_or_fit_baseline(
        cno_path,
        cno_model,
        lambda: fit_cno_direct_model(
            cno_model,
            objects["cno_train_loader"],
            objects["cno_val_loader"],
        ),
        _paper_checkpoint_metadata(
            n_train,
            "CNO-Direct",
            "one direct map from (u0,x) to every future saved state",
            (
                "upstream CNO1d architecture and default depth/block settings; "
                "out_dim is Nt-1 to represent all future snapshots"
            ),
        ),
        legacy_paths=paper_v1_checkpoint_candidates(n_train, "CNO_Direct"),
    )

    ffno_fit = _load_or_fit_baseline(
        ffno_path,
        ffno_model,
        lambda: fit_paper_ar_model(
            ffno_model,
            objects["ffno_train_loader"],
            objects["ffno_val_loader"],
            objects["U_val"],
            ffno_predict,
            name="F-FNO",
            loss_kind="mse",
            learning_rate=FFNO_LR,
            weight_decay=FFNO_WEIGHT_DECAY,
            val_pair_interval=FFNO_PAPER_VAL_PAIR_INTERVAL,
            val_rollout_interval=FFNO_PAPER_VAL_ROLLOUT_INTERVAL,
        ),
        _paper_checkpoint_metadata(
            n_train,
            "F-FNO",
            (
                "all adjacent training pairs; one-step Markov map; recursive "
                "validation and recursive test rollout from u0"
            ),
            (
                "competitive one-axis adaptation of FourierFlow's factorized "
                "2D block; the upstream repository contains no separate 1D "
                "F-FNO; uses the benchmark's periodic coordinate features"
            ),
        ),
        legacy_paths=competitive_ffno_checkpoint_candidates(n_train),
    )

    for fit, name in [
        (unet_fit, f"N{n_train}_U-Net-AR"),
        (cno_fit, f"N{n_train}_CNO-Direct"),
        (ffno_fit, f"N{n_train}_F-FNO"),
    ]:
        if fit.get("history") is not None:
            plot_training_curves(fit["history"], name)

    return {
        "n_train": int(n_train),
        "base_tag": BASE_TAG,
        "objects": objects,
        "run_seed": run_seed,
        "unet_fit": unet_fit,
        "cno_fit": cno_fit,
        "ffno_fit": ffno_fit,
        "unet_predict": unet_predict,
        "cno_predict": lambda u0: cno_model(u0),
        "ffno_predict": ffno_predict,
    }

def plot_paper_baseline_l2_per_time(stats):
    if not SAVE_PLOTS or not MAKE_L2_PER_TIME_PLOT:
        return
    out_dir = PLOT_DIR / "paper_baselines_l2_error_per_timestep"
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8, 5))
    for name in PAPER_BASELINE_NAMES:
        plt.plot(t_grid, stats[name]["mean_t"], label=f"{name} mean")
        plt.plot(t_grid, stats[name]["median_t"], "--", label=f"{name} median")
    plt.xlabel("time")
    plt.ylabel(r"$E_{L^2}$")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    path = out_dir / f"mean_median_L2_{BASE_TAG}.png"
    plt.savefig(path, dpi=PLOT_DPI)
    plt.close()
    print("Saved:", path)

def print_paper_baseline_results(stats, train_times, inference_times, parameters):
    print("\n================ PAPER-BASELINE PHYSICAL L2 RESULTS ================")
    print("Full trajectory: absolute L2(Omega x T) with the benchmark's time rule.")
    for name in PAPER_BASELINE_NAMES:
        s = stats[name]
        print(
            f"{name:<10} | full mean {s['full_l2_mean']:.6e} | "
            f"std {s['full_l2_std']:.6e} | median {s['full_l2_median']:.6e} | "
            f"final mean {s['final_l2_mean']:.6e} | "
            f"final median {s['final_l2_median']:.6e} | "
            f"params {parameters[name]:,} | train {train_times[name]:.2f}s | "
            f"inference {inference_times[name]:.4f}s"
        )
    winner = min(stats, key=lambda name: stats[name]["full_l2_mean"])
    print("Winner:", winner)
    print("====================================================================\n")
    return winner

def evaluate_paper_baselines(bundle, U_test, test_ids):
    global BASE_TAG
    BASE_TAG = bundle["base_tag"]
    n_train = bundle["n_train"]
    u0_test = np.asarray(U_test[:, :, 0], dtype=np.float32)
    nt = U_test.shape[-1]

    print("\n" + "=" * 88)
    print(
        "COMMON TEST EVALUATION: U-Net-AR, CNO-Direct, F-FNO "
        f"| N_train = {n_train}"
    )
    print("=" * 88)
    print("All models receive the exact test u0 and no exact future test state.")

    U_unet, unet_inference = _timed_prediction(
        lambda: autoregressive_rollout_generic(
            bundle["unet_fit"]["model"],
            u0_test,
            nt,
            lambda u: bundle["unet_predict"](u, training_noise=False),
            name="U-Net-AR",
            batch_size=UNET_AR_BATCH_SIZE,
            print_diagnostics=True,
        )
    )
    U_cno, cno_inference = _timed_prediction(
        lambda: direct_full_trajectory_prediction(
            bundle["cno_fit"]["model"],
            u0_test,
            nt,
            bundle["cno_predict"],
            name="CNO-Direct",
            batch_size=CNO_DIRECT_BATCH_SIZE,
        )
    )
    U_ffno, ffno_inference = _timed_prediction(
        lambda: autoregressive_rollout_generic(
            bundle["ffno_fit"]["model"],
            u0_test,
            nt,
            lambda u: bundle["ffno_predict"](u, training_noise=False),
            name="F-FNO",
            batch_size=FFNO_BATCH_SIZE,
            print_diagnostics=True,
        )
    )

    rollouts = {
        "U-Net-AR": U_unet,
        "CNO-Direct": U_cno,
        "F-FNO": U_ffno,
    }
    stats = {
        name: rollout_l2_metrics(pred, U_test)
        for name, pred in rollouts.items()
    }
    train_times = {
        "U-Net-AR": float(bundle["unet_fit"]["train_time_sec"]),
        "CNO-Direct": float(bundle["cno_fit"]["train_time_sec"]),
        "F-FNO": float(bundle["ffno_fit"]["train_time_sec"]),
    }
    inference_times = {
        "U-Net-AR": float(unet_inference),
        "CNO-Direct": float(cno_inference),
        "F-FNO": float(ffno_inference),
    }
    parameters = {
        "U-Net-AR": count_params(bundle["unet_fit"]["model"]),
        "CNO-Direct": count_params(bundle["cno_fit"]["model"]),
        "F-FNO": count_params(bundle["ffno_fit"]["model"]),
    }
    winner = print_paper_baseline_results(
        stats, train_times, inference_times, parameters
    )

    for name, pred in rollouts.items():
        print_terminal_errors(pred, U_test, name)

    plot_paper_baseline_l2_per_time(stats)
    plot_per_test_l2_curves({
        name: model_stats["all_l2_t"]
        for name, model_stats in stats.items()
    })
    plot_solution_curves(U_test, rollouts)

    rollout_path = RESULTS_DIR / (
        f"N{n_train}_paper_baseline_rollouts_{BASE_TAG}.npz"
    )
    np.savez_compressed(
        rollout_path,
        U_roll_UNET_AR=U_unet,
        U_roll_CNO_DIRECT=U_cno,
        U_roll_FFNO=U_ffno,
        U_true=U_test,
        x_grid=x_grid_np,
        t=t_grid,
        test_ids=test_ids,
        UNET_AR_l2_t=stats["U-Net-AR"]["all_l2_t"],
        CNO_DIRECT_l2_t=stats["CNO-Direct"]["all_l2_t"],
        FFNO_l2_t=stats["F-FNO"]["all_l2_t"],
        UNET_AR_full_l2=stats["U-Net-AR"]["all_full_l2"],
        CNO_DIRECT_full_l2=stats["CNO-Direct"]["all_full_l2"],
        FFNO_full_l2=stats["F-FNO"]["all_full_l2"],
        UNET_AR_final_l2=stats["U-Net-AR"]["all_final_l2"],
        CNO_DIRECT_final_l2=stats["CNO-Direct"]["all_final_l2"],
        FFNO_final_l2=stats["F-FNO"]["all_final_l2"],
    )
    print("Saved rollout arrays:", rollout_path)

    row = {
        "N_train": int(n_train),
        "winner": winner,
        "UNET_AR_full_l2_mean": stats["U-Net-AR"]["full_l2_mean"],
        "CNO_DIRECT_full_l2_mean": stats["CNO-Direct"]["full_l2_mean"],
        "FFNO_full_l2_mean": stats["F-FNO"]["full_l2_mean"],
        "UNET_AR_full_l2_std": stats["U-Net-AR"]["full_l2_std"],
        "CNO_DIRECT_full_l2_std": stats["CNO-Direct"]["full_l2_std"],
        "FFNO_full_l2_std": stats["F-FNO"]["full_l2_std"],
        "UNET_AR_full_l2_median": stats["U-Net-AR"]["full_l2_median"],
        "CNO_DIRECT_full_l2_median": stats["CNO-Direct"]["full_l2_median"],
        "FFNO_full_l2_median": stats["F-FNO"]["full_l2_median"],
        "UNET_AR_final_l2_mean": stats["U-Net-AR"]["final_l2_mean"],
        "CNO_DIRECT_final_l2_mean": stats["CNO-Direct"]["final_l2_mean"],
        "FFNO_final_l2_mean": stats["F-FNO"]["final_l2_mean"],
        "UNET_AR_final_l2_median": stats["U-Net-AR"]["final_l2_median"],
        "CNO_DIRECT_final_l2_median": stats["CNO-Direct"]["final_l2_median"],
        "FFNO_final_l2_median": stats["F-FNO"]["final_l2_median"],
        "UNET_AR_best_val_rollout_l2": float(
            bundle["unet_fit"]["best_val_l2"]
        ),
        "CNO_DIRECT_best_val_l2": float(bundle["cno_fit"]["best_val_l2"]),
        "FFNO_best_val_rollout_l2": float(bundle["ffno_fit"]["best_val_l2"]),
        "UNET_AR_parameters": parameters["U-Net-AR"],
        "CNO_DIRECT_parameters": parameters["CNO-Direct"],
        "FFNO_parameters": parameters["F-FNO"],
        "UNET_AR_train_time_sec": train_times["U-Net-AR"],
        "CNO_DIRECT_train_time_sec": train_times["CNO-Direct"],
        "FFNO_train_time_sec": train_times["F-FNO"],
        "UNET_AR_inference_time_sec": inference_times["U-Net-AR"],
        "CNO_DIRECT_inference_time_sec": inference_times["CNO-Direct"],
        "FFNO_inference_time_sec": inference_times["F-FNO"],
        "UNET_AR_model_calls_per_trajectory": int(nt - 1),
        "CNO_DIRECT_model_calls_per_trajectory": 1,
        "FFNO_model_calls_per_trajectory": int(nt - 1),
        "AR_training_pairs": bundle["objects"]["n_pair_train"],
        "base_tag": BASE_TAG,
        "rollout_path": str(rollout_path),
    }

    summary = {
        "task": EXPERIMENT_NAME,
        "suite": "paper_baselines",
        "protocol_version": PAPER_BASELINE_VERSION,
        "data_path": str(DATA_PATH),
        "N_train": int(n_train),
        "N_val": N_VAL,
        "N_test": N_TEST,
        "Nx": Nx,
        "Nt": Nt,
        "split_mode": SPLIT_MODE,
        "split_seed": SPLIT_SEED,
        "run_seed": bundle["run_seed"],
        "test_information": "only exact u0; no exact future test state",
        "metric": {
            "name": "absolute L2(Omega x T)",
            "dx": DX,
            "dt_snapshot": DT_SNAPSHOT,
            "trapezoidal_time_rule": USE_TRAPEZOIDAL_TIME_L2,
            "includes_t0": INCLUDE_T0_IN_FULL_L2,
        },
        "protocols": {
            "U-Net-AR": (
                "one-step map trained on all adjacent pairs and recursively "
                "applied from u0"
            ),
            "CNO-Direct": (
                "non-autoregressive upstream CNO1d mapping (u0,x) to all "
                "future snapshots in one call"
            ),
            "F-FNO": (
                "one-step factorized Fourier map trained on all adjacent pairs "
                "and recursively applied from u0"
            ),
        },
        "provenance": {
            "U-Net-AR": _paper_checkpoint_metadata(
                n_train, "U-Net-AR", "see protocols", "see checkpoint"
            )["source"],
            "CNO-Direct": _paper_checkpoint_metadata(
                n_train, "CNO-Direct", "see protocols", "see checkpoint"
            )["source"],
            "F-FNO": _paper_checkpoint_metadata(
                n_train, "F-FNO", "see protocols", "see checkpoint"
            )["source"],
        },
        "metrics": row,
    }
    summary_path = RESULTS_DIR / (
        f"N{n_train}_paper_baselines_summary_{BASE_TAG}.json"
    )
    with open(summary_path, "w") as stream:
        json.dump(summary, stream, indent=2)
    print("Saved summary:", summary_path)
    return row

def plot_paper_baseline_sample_sweep(rows):
    if not SAVE_PLOTS:
        return
    rows = sorted(rows, key=lambda row: row["N_train"])
    ns = [row["N_train"] for row in rows]
    for suffix, ylabel in [
        ("full_l2_mean", "mean absolute full-trajectory L2"),
        ("final_l2_mean", "mean absolute final-time L2"),
    ]:
        plt.figure(figsize=(8, 5))
        for key, label in [
            ("UNET_AR", "U-Net-AR"),
            ("CNO_DIRECT", "CNO-Direct"),
            ("FFNO", "F-FNO"),
        ]:
            plt.plot(
                ns,
                [row[f"{key}_{suffix}"] for row in rows],
                marker="o",
                label=label,
            )
        plt.xlabel("number of training trajectories")
        plt.ylabel(ylabel)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        path = PLOT_DIR / f"paper_baselines_sample_sweep_{suffix}.png"
        plt.savefig(path, dpi=PLOT_DPI)
        plt.close()
        print("Saved:", path)

def print_paper_baseline_sample_summary(rows):
    print("\n================ PAPER BASELINE SAMPLE-SIZE SWEEP: FULL L2 ================")
    for row in sorted(rows, key=lambda item: item["N_train"]):
        print(
            f"N={row['N_train']:4d} | "
            f"U-Net-AR {row['UNET_AR_full_l2_mean']:.6e} | "
            f"CNO-Direct {row['CNO_DIRECT_full_l2_mean']:.6e} | "
            f"F-FNO {row['FFNO_full_l2_mean']:.6e} | "
            f"winner {row['winner']}"
        )
    print("=========================================================================\n")
    print("================ PAPER BASELINE SAMPLE-SIZE SWEEP: FINAL L2 ===============")
    for row in sorted(rows, key=lambda item: item["N_train"]):
        final_scores = {
            "U-Net-AR": row["UNET_AR_final_l2_mean"],
            "CNO-Direct": row["CNO_DIRECT_final_l2_mean"],
            "F-FNO": row["FFNO_final_l2_mean"],
        }
        print(
            f"N={row['N_train']:4d} | "
            f"U-Net-AR {row['UNET_AR_final_l2_mean']:.6e} | "
            f"CNO-Direct {row['CNO_DIRECT_final_l2_mean']:.6e} | "
            f"F-FNO {row['FFNO_final_l2_mean']:.6e} | "
            f"winner {min(final_scores, key=final_scores.get)}"
        )
    print("=========================================================================\n")

def make_synthetic_smoke_dataset():
    """Small smooth periodic trajectories used only to verify the code path."""
    n_total = N_TRAIN_POOL + N_VAL + N_TEST
    nx = max(TARGET_NX, 64)
    nt = env_int("SMOKE_NT", 9)
    x = np.linspace(0.0, 1.0, nx, endpoint=False, dtype=np.float32)
    t = np.linspace(0.0, 1.0, nt, dtype=np.float32)
    rng = np.random.default_rng(SEED)
    U = np.zeros((n_total, nx, nt), dtype=np.float32)
    for sample in range(n_total):
        amplitudes = rng.normal(size=4).astype(np.float32)
        phases = rng.uniform(0.0, 2.0 * np.pi, size=4).astype(np.float32)
        for ti, time_value in enumerate(t):
            state = np.zeros(nx, dtype=np.float32)
            for mode in range(1, 5):
                state += (
                    amplitudes[mode - 1]
                    * np.exp(-0.08 * mode * mode * float(time_value))
                    * np.sin(
                        2.0 * np.pi * mode * (x - 0.12 * float(time_value))
                        + phases[mode - 1]
                    )
                )
            U[sample, :, ti] = state
    print("SMOKE_TEST=True: using generated periodic trajectories", U.shape)
    return U, x, t

def main_paper_baselines():
    print("\nExperiment")
    print("----------")
    print("name              :", EXPERIMENT_NAME)
    print("suite             : paper_baselines")
    print("protocol version  :", PAPER_BASELINE_VERSION)
    print("data path         :", DATA_PATH)
    print("output root       :", OUTPUT_ROOT)
    print("cache root        :", CACHE_ROOT)
    print("device            :", device)
    print("ALL_SAMPLE_SIZES  :", ALL_SAMPLE_SIZES)
    print("RUN SAMPLE_SIZES  :", SAMPLE_SIZES)
    print("N_TRAIN_POOL      :", N_TRAIN_POOL)
    print("N_VAL / N_TEST    :", N_VAL, N_TEST)
    print("TARGET_NX         :", TARGET_NX)
    print("TIME_SUBSAMPLE    :", TIME_SUBSAMPLE)
    print("EPOCHS            :", EPOCHS)
    print("FORCE_RETRAIN     :", FORCE_RETRAIN)
    print("SMOKE_TEST        :", SMOKE_TEST)

    if SMOKE_TEST:
        U_all, x, t = make_synthetic_smoke_dataset()
    else:
        U_all, x, t = load_hdf5_1d_dataset(DATA_PATH)
        U_all, x = downsample_space_stride(U_all, x, TARGET_NX)
    set_geometry_from_arrays(U_all.shape, x, t)

    U_train_pool, train_ids, U_val, val_ids, U_test, test_ids = (
        make_train_val_test_splits(U_all)
    )
    split_path = DATA_DIR / "train_val_test_split.npz"
    save_split_dataset_npz(
        split_path,
        U_train_pool,
        train_ids,
        U_val,
        val_ids,
        U_test,
        test_ids,
        x,
        t,
    )

    if RUN_EXACT_HASH_OVERLAP_CHECK:
        check_exact_hash_overlap(U_train_pool, U_val, "train_pool", "val")
        check_exact_hash_overlap(U_train_pool, U_test, "train_pool", "test")
        check_exact_hash_overlap(U_val, U_test, "val", "test")
    if RUN_DISTANCE_OVERLAP_CHECK:
        check_dataset_overlap(
            U_train_pool, U_test, "train_pool", "test", tol=OVERLAP_TOL
        )
        check_dataset_overlap(U_val, U_test, "val", "test", tol=OVERLAP_TOL)

    rows = []
    for n_train in SAMPLE_SIZES:
        bundle = train_paper_baselines_for_sample_size(
            n_train, U_train_pool, U_val
        )
        rows.append(evaluate_paper_baselines(bundle, U_test, test_ids))
        del bundle
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    rows = sorted(rows, key=lambda row: row["N_train"])
    summary_path = RESULTS_DIR / "paper_baselines_sample_size_summary.json"
    with open(summary_path, "w") as stream:
        json.dump(rows, stream, indent=2)
    print("Saved sample-size summary:", summary_path)
    plot_paper_baseline_sample_sweep(rows)
    print_paper_baseline_sample_summary(rows)



## 4. Run the complete benchmark

This is a substantial experiment. Results and checkpoints are saved after every training size. If Colab disconnects, reconnect, rerun the configuration and definition cells, and execute this cell again with `FORCE_RETRAIN=0`.

In [ ]:
main_paper_baselines()


## 5. Optional: release the Colab runtime

Run only after the preceding cell has finished and the Drive files are visible.

In [ ]:
from google.colab import runtime
runtime.unassign()
